In [14]:
import pandas as pd
import yaml
import geodatasets
import geopandas as gpd
import math
import rasterio
import os
import openpyxl
import pygwalker as pyg
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import re
import glob
import sys
import textwrap
from shapely.geometry import Point
from datetime import datetime
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from matplotlib.transforms import blended_transform_factory
from matplotlib.gridspec import GridSpec
from io import BytesIO
from IPython.display import Image, display as ipy_display
from rasterio.windows import Window
from tqdm import tqdm

In [2]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

now = datetime.now()
today = now.strftime("%Y%m%d")

pd.set_option('display.max_columns', None)

In [3]:
# Assigns climate domain column
def assign_climate_domain(df):

    cont_eco = df["continent_ecozone"]

    df["climate_domain"] = np.select(
        [
            cont_eco.str.contains("boreal", case=False, na=False) | cont_eco.str.contains("polar", case=False, na=False),
            cont_eco.str.contains("temperate", case=False, na=False),
            cont_eco.str.contains("tropical", case=False, na=False) | cont_eco.str.contains("subtropical", case=False, na=False),
        ],
        [
            "Boreal",
            "Temperate",
            "Subtropical/tropical",
        ],
        default="Other"
    )

    return df

In [4]:
# Fills in the intervening years of the multi-year intervals for soil data.
# Basically, duplicates values from end-of-interval years for all the preceding years in the interval,
# but only for that row's own nominal interval. Missing intervals stay blank.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c2fe30-7004-8328-9f6d-1c231ba04e20
def fill_in_soil_years(df, group_cols):

    # Aggregate first
    df = (
        df
        .groupby(group_cols, dropna=False, as_index=False)
        .agg({
            "flux_Mg_CO2e_yr": "sum",
            "area_ha": "sum"
        })
    )

    # Sort by timeseries + year
    df = df.sort_values(series_cols + ["year"]).reset_index(drop=True)

    year_vals = df["year"].astype(int)

    # Maximum (final) year in the dataset
    last_year = year_vals.max()

    # Nominal interval length for each endpoint year:
    # - years divisible by 5 -> 5-year interval
    # - final year in dataset -> interval from 2021 to last_year
    # - everything else defaults to 1 year unless you add more rules
    interval_len = np.where(
        year_vals.eq(last_year),
        last_year - 2021 + 1,
        np.where(year_vals.mod(5).eq(0), 5, 1)
    )

    # Start year comes only from the row's own nominal interval,
    # not from the previous observed year.
    start_year = (year_vals - interval_len + 1).astype(int).to_numpy()
    end_year = year_vals.to_numpy()

    # Number of repetitions for each interval
    n_rep = end_year - start_year + 1

    # Duplicate rows
    expanded = df.loc[df.index.repeat(n_rep)].copy()

    # Assign correct years
    expanded["year"] = np.concatenate([
        np.arange(s, e + 1) for s, e in zip(start_year, end_year)
    ])

    expanded = expanded.reset_index(drop=True)

    # If the source data ends before the end of the vegetation timeseries, extendyears through 2024 to match vegetation timeseries
    if last_year < cn.years_annual[-1]:
        rows_2022 = expanded[expanded["year"] == 2022].copy()

        rows_2023 = rows_2022.copy()
        rows_2023["year"] = 2023

        rows_2024 = rows_2022.copy()
        rows_2024["year"] = 2024

        expanded = pd.concat([expanded, rows_2023, rows_2024], ignore_index=True)
        expanded = expanded.sort_values(series_cols + ["year"]).reset_index(drop=True)

    return expanded

In [97]:
# Converts long-format df to wide-format df
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69fe3161-edb8-832e-a90d-d9e75e4012d3
def create_wide_df(combined_df, value_field):

    print(f"Converting combined table from long to wide: {uu.timestr()}")

    # Columns to use and to not use as contextual layers. Drops gas because it's implicit in analysis_layer.
    id_cols = [
        c for c in combined_df.columns
        if c not in ["analysis_layer", value_field, "density__Mg_ha", "area_ha", "gas", "LULUCF_component"]
    ]

    # Reshapes from long to wide, with value and area_ha for each analysis_layer.
    # ChatGPT says this is safer for giant tables than using pivot_table
    print(f"  Grouping and summing: {uu.timestr()}")
    wide = (
        combined_df
        .groupby(id_cols + ["analysis_layer"], observed=True, sort=False, dropna=False)[[value_field]]
        # .groupby(id_cols + ["analysis_layer"], observed=True, sort=False)[[value_field, "area_ha"]]
        .sum()
        .unstack("analysis_layer")
        .fillna(0)
    )

    # Appends __value or __area_ha to each analysis layer
    print(f"  Cleaning columns: {uu.timestr()}")
    wide.columns = [
        f"{analysis_layer}__value"
        for _, analysis_layer in wide.columns
    ]
    wide = wide.reset_index()

    # Reorders columns so that __value fields are before __area_ha fields
    # value_cols = sorted([c for c in wide.columns if c.endswith("__value")])
    # wide = wide[id_cols + value_cols]
    area_cols = sorted([c for c in wide.columns if c.endswith("__area_ha")])
    wide = wide[id_cols + value_cols + area_cols]
    combined_wide_df = wide.reset_index(drop=True)


    print(f"  Making summative soil columns: {uu.timestr()}")

    # Sum all soil columns. This has to come before the organic soil summation. 
    soil_cols = [c for c in combined_wide_df.columns if "soil" in c]
    
    combined_wide_df["soil__all_gases__MgCO2e__value"] = (
        combined_wide_df[soil_cols]
        .sum(axis=1)
    )
    
    # Sum all organic soil columns
    organic_cols = [c for c in combined_wide_df.columns if "organic" in c]
    
    combined_wide_df["organic_soil__all_gases__MgCO2e__value"] = (
        combined_wide_df[organic_cols]
        .sum(axis=1)
    )

    return combined_wide_df

In [237]:
# Combines 10x10 deg tile veg parquets into global long and wide tables using specified contextual layers
def aggreg_veg_parquets(parquet_files, agg_layers):

    # Sums value and area_ha by contextual layer combinations for each tile
    print("Simplifying each tile.")
    tile_aggs = []
    for f in tqdm(parquet_files):
        df_tile = pd.read_parquet(f, columns=agg_layers + ['value', 'area_ha'])
        agg = (
            df_tile
            .groupby(agg_layers, dropna=False)[['value', 'area_ha']]
            .sum()
            .reset_index()
        )
        tile_aggs.append(agg)

    # Sums across tiles, consolidating across combinations of contextual layers that repeat across tiles 
    print("Summing and simplifying all tiles")
    sums = (
        pd.concat(tile_aggs, ignore_index=True)
        .pipe(lambda df: df[~df['analysis_layer'].isin([
            'AGC_emission_factor_CO2_only__fraction',
            'removal_factor__AGC__MgC',
            'carbon_density__non_soil__MgC_ha',
        ])])
        .groupby(agg_layers, dropna=False)[['value', 'area_ha']]
        .sum()
        .reset_index()
        .sort_values(agg_layers)
        .reset_index(drop=True)
    )
    sums.rename(columns={'value': 'zonal_stats_sum'}, inplace=True)
    # display(sums)

    # Converts from long to wide format with all contextual columns
    print("Converting table to wide format")
    contextual_cols = [c for c in agg_layers if c != 'analysis_layer']

    sums_wide = (
        sums
        .set_index(contextual_cols + ['analysis_layer'])[['zonal_stats_sum', 'area_ha']]
        .unstack('analysis_layer')
    )

    # Flatten MultiIndex columns: flux cols keep layer name, area cols get __area_ha suffix
    sums_wide.columns = [
        layer if metric == 'zonal_stats_sum' else f'{layer}__area_ha'
        for metric, layer in sums_wide.columns
    ]
    sums_wide = sums_wide.reset_index()

    flux_cols = [c for c in sums_wide.columns if c not in contextual_cols and not c.endswith('__area_ha')]
    area_cols = [c for c in sums_wide.columns if c.endswith('__area_ha')]
    sums_wide[flux_cols] = sums_wide[flux_cols].fillna(0).round(0).astype(int)
    sums_wide[area_cols] = sums_wide[area_cols].fillna(0).round(0).astype(int)
    
    # display(sums_wide)

    return sums, sums_wide

In [214]:
# Compares chunk stats and zonal stats
def compare_chunk_and_zonal_stats(chunk_tile_agg, zonal_table):

    # Merges chunk stats and zonal stats
    comparison = pd.merge(
        chunk_tile_agg,
        zonal_table.drop(columns='area_ha'),
        on=['analysis_layer', 'year'],
        how='outer',
        indicator=True
    )
    
    only_in_chunk = comparison[comparison['_merge'] == 'left_only']
    only_in_flox  = comparison[comparison['_merge'] == 'right_only']
    print(f"Variable×year only in chunk stats: {len(only_in_chunk)}")
    print(f"Variable×year only in flox stats:  {len(only_in_flox)}")

    comparison = comparison[comparison['_merge'] == 'both'].drop(columns='_merge')
    display(comparison)

    comparison['abs_diff'] = comparison['zonal_stats_sum'] - comparison['chunk_stats_sum']
    comparison['pct_diff'] = np.where(
        comparison['chunk_stats_sum'] == 0,
        np.where(comparison['zonal_stats_sum'] == 0, 0.0, np.inf),
        (comparison['zonal_stats_sum'] - comparison['chunk_stats_sum']) / comparison['chunk_stats_sum'].abs() * 100
    )

    MISMATCH_THRESHOLD_PCT = 0.1

    mismatches = (
        comparison[comparison['pct_diff'].abs() > MISMATCH_THRESHOLD_PCT]
        .sort_values('pct_diff', key=lambda s: s.abs(), ascending=False)
        .reset_index(drop=True)
    )

    print(f"Total variable×year combinations compared: {len(comparison)}")
    print(f"Mismatches (|pct_diff| > {MISMATCH_THRESHOLD_PCT}%):  {len(mismatches)}")
    print(f"Matching:                                   {len(comparison) - len(mismatches)}")

    layer_summary = (
        mismatches
        .groupby('analysis_layer')
        .agg(
            n_mismatches=('year', 'count'),
            max_abs_pct_diff=('pct_diff', lambda x: x.abs().max()),
            years=('year', lambda x: sorted(x.unique()))
        )
        .sort_values('max_abs_pct_diff', ascending=False)
        .reset_index()
    )
    print(f"\nAnalysis layers with at least one mismatch: {len(layer_summary)}")

    return comparison, mismatches, layer_summary

In [ ]:
# Context columns that define a unique timeseries. Does not include year.
series_cols = [
    "analysis_layer",
    "adm0", "country_name", "region_L1", "region_L2_L3",
    "land_state_node", "land_state", "land_state_meaning",
    "WDPA", "WDPA_type", 
    "cont_eco", "continent", "ecozone", "continent_ecozone", "climate_domain",
    "Landmark",
    "starting_composite_primary_forest",
    "tile_id",
    "land_state_broad_class",
    "land_state_detailed_class",
    "WDPA_high_protection",
    "tall_veg_type",
    "LULUCF_component",
    "gas",
    "watershed", "watershed_name", 
    "KBA",
    "drivers_of_TCL_1_km", "driver_1km_text"
]

# Contextual columns for grouping in export csvs. Includes year.
export_csv_group_cols = [
    "analysis_layer",
    "adm0", "country_name", "region_L1", "region_L2_L3",
    "cont_eco", "continent", "continent_ecozone", "climate_domain",
    "year",
    "land_state_broad_class",
    "land_state_detailed_class",
    "tall_veg_type",
    "LULUCF_component"
]

### Input table paths

In [43]:
# Vegetation zonal stats output folder (separate parquet for each tile-- too large to combine into a single parquet)
zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__full_run_with_Canada_rerun__20260603/'

In [225]:
# SOC (including mineral soil) zonal stats output
SOC_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/SOC_v{cn.SOC_model_version_underscore}_standard_global/'
SOC_parquet_name = f'SOC_zonal_stats_v{cn.SOC_model_version_underscore}_20260531_21_15_30.parquet'

In [ ]:
# Organic soil zonal stats output
# org_soil_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/organic_soil_v{cn.organic_soil_model_version_underscore}_standard_global__20260512/'
org_soil_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/organic_soil_v1_0_0_standard_global__20260512/'
# Erin's v1.0.0 run only has the 2024 interval, but I need the preceding interval as well, so I just copied the final interval into the 2016-2020 period
org_soil_csv_name = 'stats_for_lulucf_paper__from_Erin_Glen_via_Slack_20260512__final_period_copied_back.csv'  

In [7]:
# LULUCF outputs
LULUCF_zonal_stats_folder = Path(f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v{cn.LULUCF_model_version_underscore}__veg_v{cn.veg_model_version_underscore}__minsoil_v{cn.SOC_model_version_underscore}__orgsoil_v{cn.organic_soil_model_version_underscore}/')
LULUCF_zonal_stats_folder.mkdir(parents=True, exist_ok=True)
LULUCF_zonal_stats_folder

PosixPath('/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1')

### Full dataframes with basic standardization of organic soil and SOC density to vegetation df

In [171]:
# Vegetation chunk stats output (for comparison with zonal stats)
chunk_stats_folder = f'/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/parquet_20260131_10_37_46__KEEP/'
gross_outputs_parquet = f'{chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__gross_outputs_1x1.parquet'
net_outputs_parquet = f'{chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__net_outputs_1x1.parquet'
chunk_stats_gross = pd.read_parquet(gross_outputs_parquet)
chunk_stats_net = pd.read_parquet(net_outputs_parquet)
# Combine gross and net chunk stats, aggregate to 10x10 deg tile level
chunk_stats_combined = pd.concat([chunk_stats_gross, chunk_stats_net], ignore_index=True)

,tile_id,analysis_layer,year,chunk_stats_sum
0,00N_000E,gross_emissions__AGC__MgCO2,2016,6.803591e+05
1,00N_000E,gross_emissions__AGC__MgCO2,2017,1.055552e+06
2,00N_000E,gross_emissions__AGC__MgCO2,2018,8.115753e+05
3,00N_000E,gross_emissions__AGC__MgCO2,2019,9.394097e+05
4,00N_000E,gross_emissions__AGC__MgCO2,2020,1.174674e+06
...,...,...,...,...
64075,80N_180W,net_flux__litter_C__MgCO2,2020,0.000000e+00
64076,80N_180W,net_flux__litter_C__MgCO2,2021,0.000000e+00
64077,80N_180W,net_flux__litter_C__MgCO2,2022,0.000000e+00
64078,80N_180W,net_flux__litter_C__MgCO2,2023,0.000000e+00


In [100]:
veg_parquet_files = sorted(glob.glob(f'{zonal_stats_folder}*.parquet'))
# veg_parquet_files_subset = veg_parquet_files[0:5]
print(f"Found {len(parquet_files)} parquet files")

Found 356 parquet files


### Test tile comparison for chunk stats vs. zonal stats: 00N_110E

In [207]:
# Chunk stats for just 00N_110E

chunk_tile_agg = (
    chunk_stats_combined
    .groupby(['tile_id', 'pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
chunk_tile_agg['year'] = chunk_tile_agg['year'].astype(int)
chunk_tile_agg.rename(columns={'variable': 'analysis_layer', 'tile_sum': 'chunk_stats_sum'}, inplace=True)
chunk_tile_agg['analysis_layer'] = chunk_tile_agg['analysis_layer'].str.replace('_ha_yr', '', regex=False)  # Removes _ha_yr to match zonal stats output
chunk_tile_agg = chunk_tile_agg[chunk_tile_agg['tile_id'] == "00N_110E"]
chunk_tile_agg = chunk_tile_agg.drop(columns='tile_id')
chunk_tile_agg

,analysis_layer,year,chunk_stats_sum
3060,gross_emissions__AGC__MgCO2,2016,2.201096e+08
3061,gross_emissions__AGC__MgCO2,2017,9.068073e+07
3062,gross_emissions__AGC__MgCO2,2018,1.114415e+08
3063,gross_emissions__AGC__MgCO2,2019,1.264043e+08
3064,gross_emissions__AGC__MgCO2,2020,9.580624e+07
...,...,...,...
3235,net_flux__litter_C__MgCO2,2020,3.817308e+05
3236,net_flux__litter_C__MgCO2,2021,2.291734e+05
3237,net_flux__litter_C__MgCO2,2022,2.738163e+05
3238,net_flux__litter_C__MgCO2,2023,5.905036e+05


In [238]:
# Simple aggregation of zonal stats for 00N_110E as test tile

agg_layers = ['analysis_layer', 'year']

zonal_sum_basic_00N_110E, zonal_sum_wide_basic_00N_110E = aggreg_veg_parquets(veg_parquet_files[17:18], agg_layers)  # 00N_110E
display(zonal_sum_basic_00N_110E)
display(zonal_sum_wide_basic_00N_110E)

Simplifying each tile.


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.90it/s]

Summing and simplifying all tiles
Converting table to wide format


,analysis_layer,year,zonal_stats_sum,area_ha
0,gross_emissions__AGC__MgCO2,2016,2.201096e+08,1.190968e+06
1,gross_emissions__AGC__MgCO2,2017,9.068074e+07,5.578561e+05
2,gross_emissions__AGC__MgCO2,2018,1.114415e+08,7.134454e+05
3,gross_emissions__AGC__MgCO2,2019,1.264043e+08,1.153870e+06
4,gross_emissions__AGC__MgCO2,2020,9.580624e+07,6.251664e+05
...,...,...,...,...
175,net_flux__litter_C__MgCO2,2020,3.817307e+05,2.396489e+06
176,net_flux__litter_C__MgCO2,2021,2.291734e+05,2.649654e+06
177,net_flux__litter_C__MgCO2,2022,2.738164e+05,2.935543e+06
178,net_flux__litter_C__MgCO2,2023,5.905036e+05,3.253276e+06


,year,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2,gross_emissions__AGC__MgCO2__area_ha,gross_emissions__BGC__MgCO2__area_ha,gross_emissions__CH4__MgCO2e__area_ha,gross_emissions__N2O__MgCO2e__area_ha,gross_emissions__all_C_pools__CO2_only__MgCO2__area_ha,gross_emissions__all_C_pools__all_gases__MgCO2e__area_ha,gross_emissions__all_C_pools__non_CO2_only__MgCO2e__area_ha,gross_emissions__deadwood_C__MgCO2__area_ha,gross_emissions__litter_C__MgCO2__area_ha,gross_removals__AGC__MgCO2__area_ha,gross_removals__BGC__MgCO2__area_ha,gross_removals__all_C_pools__MgCO2__area_ha,gross_removals__deadwood_C__MgCO2__area_ha,gross_removals__litter_C__MgCO2__area_ha,net_flux__AGC__MgCO2__area_ha,net_flux__BGC__MgCO2__area_ha,net_flux__all_C_pools__CO2_only__MgCO2__area_ha,net_flux__all_C_pools__all_gases__MgCO2e__area_ha,net_flux__deadwood_C__MgCO2__area_ha,net_flux__litter_C__MgCO2__area_ha
0,2016,220109648,57282324,50047,18165,282227040,282295232,68212,4136428,698634,-134113976,-47378740,-181663504,-151034,-19754,85995664,9903585,100563520,100631736,3985394,678880,1190968,1157365,34798,34798,1190968,1211341,34798,349259,349259,31997580,31977928,31997580,537585,537585,32382262,32350050,32382262,32402634,886845,886845
1,2017,90680744,26480458,33241,15949,120630992,120680184,49190,2965724,504072,-132927528,-40322940,-173506656,-224670,-31510,-42246788,-13842480,-52875652,-52826464,2741053,472562,557856,528991,51968,51968,557856,598042,51968,270541,270541,31972530,31940624,31972530,880522,880522,32212756,32186368,32212756,32252942,1151063,1151063
2,2018,111441504,31383544,200411,80620,147151536,147432560,281031,3695324,631153,-137146544,-42424812,-180000960,-373816,-55785,-25705038,-11041267,-32849432,-32568400,3321508,575368,713445,609052,222452,222452,713445,846535,222452,343768,343768,32214370,32161454,32214370,1354679,1354679,32518900,32426410,32518900,32651990,1698447,1698447
3,2019,126404312,33029894,1289054,452872,163484928,165226864,1741926,3458704,592025,-138888576,-43829108,-183275680,-484529,-73472,-12484261,-10799213,-19790742,-18048818,2974175,518553,1153870,677782,918177,918177,1153870,1588530,918177,365126,365126,32030772,31952614,32030772,1708824,1708824,32669820,32244766,32669820,33104478,2073951,2073951
4,2020,95806240,26775816,25963,9424,125877664,125913048,35387,2819825,475778,-142410656,-42657604,-185776608,-614315,-94048,-46604404,-15881788,-59898948,-59863560,2205510,381731,625166,525885,20331,20331,625166,638079,20331,265834,265834,32413824,32362128,32413824,2130655,2130655,32692548,32597646,32692548,32705462,2396489,2396489
5,2021,72104176,20076744,14398,6523,94519768,94540688,20921,1998520,340327,-144626736,-42489968,-187943296,-715431,-111154,-72522560,-22413224,-93423512,-93402592,1283089,229173,499368,382818,19486,19486,499368,514158,19486,175064,175064,32569864,32540770,32569864,2474591,2474591,32818860,32707228,32818860,32833650,2649654,2649654
6,2022,82906288,23218966,12539,5446,108882664,108900648,17984,2355948,401458,-146618080,-43845656,-191404032,-812648,-127641,-63711788,-20626690,-82521360,-82503376,1543300,273816,469153,427326,14544,14544,469154,479486,14544,200783,200783,32685604,32668210,32685604,2734760,2734760,32851936,32819634,32851936,32862268,2935543,2935543
7,2023,148694576,40697780,448201,171127,194414272,195033600,619328,4294587,727327,-147642224,-47798964,-196444688,-866683,-136823,1052355,-710

In [221]:
# Compares chunk stats and zonal stats for a tile (previously selected)

comparison, mismatches, tile_summary = compare_chunk_and_zonal_stats(chunk_tile_agg, zonal_sum_basic_00N_110E)
# display(comparison)
display(mismatches)
display(tile_summary)

Variable×year only in chunk stats: 180
Variable×year only in flox stats:  180


,analysis_layer,year,chunk_stats_sum,zonal_stats_sum


Total variable×year combinations compared: 0
Mismatches (|pct_diff| > 0.1%):  0
Matching:                                   0

Analysis layers with at least one mismatch: 0


,analysis_layer,year,chunk_stats_sum,zonal_stats_sum,abs_diff,pct_diff


,analysis_layer,n_mismatches,max_abs_pct_diff,years


### Global comparison for chunk stats vs. zonal stats with few contextual layers

In [218]:
# Chunk stats for all tiles
chunk_tile_agg = (
    chunk_stats_combined
    .groupby(['pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
chunk_tile_agg['year'] = chunk_tile_agg['year'].astype(int)
chunk_tile_agg.rename(columns={'variable': 'analysis_layer', 'tile_sum': 'chunk_stats_sum'}, inplace=True)
chunk_tile_agg

,analysis_layer,year,chunk_stats_sum
0,gross_emissions__AGC__MgCO2_ha_yr,2016,8.535058e+09
1,gross_emissions__AGC__MgCO2_ha_yr,2017,7.968987e+09
2,gross_emissions__AGC__MgCO2_ha_yr,2018,8.017658e+09
3,gross_emissions__AGC__MgCO2_ha_yr,2019,8.496905e+09
4,gross_emissions__AGC__MgCO2_ha_yr,2020,9.722788e+09
...,...,...,...
175,net_flux__litter_C__MgCO2_ha_yr,2020,2.525365e+07
176,net_flux__litter_C__MgCO2_ha_yr,2021,2.392749e+07
177,net_flux__litter_C__MgCO2_ha_yr,2022,2.129800e+07
178,net_flux__litter_C__MgCO2_ha_yr,2023,1.750981e+07


In [219]:
%%time

# Simple aggregation of zonal stats globally

agg_layers = ['analysis_layer', 'year']

zonal_sum_basic_00N_110E, zonal_sum_wide_basic_00N_110E = aggreg_veg_parquets(veg_parquet_files, agg_layers)  
# zonal_sum_basic_00N_110E

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 356/356 [01:09<00:00,  5.15it/s]


In [220]:
# Compares chunk stats and zonal stats for a tile (previously selected)

comparison, mismatches, tile_summary = compare_chunk_and_zonal_stats(chunk_tile_agg, zonal_sum_basic_00N_110E)
display(mismatches)
display(tile_summary)

Variable×year only in chunk stats: 180
Variable×year only in flox stats:  180


,analysis_layer,year,chunk_stats_sum,zonal_stats_sum


Total variable×year combinations compared: 0
Mismatches (|pct_diff| > 0.1%):  0
Matching:                                   0

Analysis layers with at least one mismatch: 0


,analysis_layer,year,chunk_stats_sum,zonal_stats_sum,abs_diff,pct_diff


,analysis_layer,n_mismatches,max_abs_pct_diff,years


### Global zonal stats with lots of contextual layers

In [224]:
%%time

# Aggregation of zonal stats by multiple contextual layers

agg_layers = ['analysis_layer', 'year', 
              'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type',
              'adm0', 'country_name', 'region_L1', 'region_L2_L3',
              'continent', 'ecozone', 'continent_ecozone', 'climate_domain'
             ]

zonal_sum_state_adm_conteco, zonal_sum_wide_state_adm_conteco = aggreg_veg_parquets(veg_parquet_files, agg_layers)
display(zonal_sum_state_adm_conteco.head())
display(zonal_sum_wide_state_adm_conteco.head())

,analysis_layer,year,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,adm0,country_name,region_L1,region_L2_L3,continent,ecozone,continent_ecozone,climate_domain,zonal_stats_sum,area_ha
0,gross_emissions__AGC__MgCO2,2016,12100000,Temporary loss of mangroves,tree,tree_loss,mangrove,AGO,Angola,Africa,Middle Africa,Africa,Tropical dry forest,Africa-Tropical dry forest,Subtropical/tropical,6183.112305,132.871826
1,gross_emissions__AGC__MgCO2,2016,12100000,Temporary loss of mangroves,tree,tree_loss,mangrove,ARE,United Arab Emirates,Asia,Western Asia,Asia,Tropical desert,Asia-Tropical desert,Subtropical/tropical,4.938448,3.287795
2,gross_emissions__AGC__MgCO2,2016,12100000,Temporary loss of mangroves,tree,tree_loss,mangrove,ATG,Antigua and Barbuda,North America,Caribbean,America South,Tropical dry forest,America South-Tropical dry forest,Subtropical/tropical,62.056431,3.086465
3,gross_emissions__AGC__MgCO2,2016,12100000,Temporary loss of mangroves,tree,tree_loss,mangrove,ATG,Antigua and Barbuda,North America,Caribbean,America South,Tropical moist deciduous forest,America South-Tropical moist deciduous forest,Subtropical/tropical,7.413018,0.073651
4,gross_emissions__AGC__MgCO2,2016,12100000,Temporary loss of mangroves,tree,tree_loss,mangrove,AUS,Australia,Oceania,Australia and New Zealand,Asia,Subtropical desert,Asia-Subtropical desert,Subtropical/tropical,1772.811401,356.751007


,year,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,adm0,country_name,region_L1,region_L2_L3,continent,ecozone,continent_ecozone,climate_domain,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2,gross_emissions__AGC__MgCO2__area_ha,gross_emissions__BGC__MgCO2__area_ha,gross_emissions__CH4__MgCO2e__area_ha,gross_emissions__N2O__MgCO2e__area_ha,gross_emissions__all_C_pools__CO2_only__MgCO2__area_ha,gross_emissions__all_C_pools__all_gases__MgCO2e__area_ha,gross_emissions__all_C_pools__non_CO2_only__MgCO2e__area_ha,gross_emissions__deadwood_C__MgCO2__area_ha,gross_emissions__litter_C__MgCO2__area_ha,gross_removals__AGC__MgCO2__area_ha,gross_removals__BGC__MgCO2__area_ha,gross_removals__all_C_pools__MgCO2__area_ha,gross_removals__deadwood_C__MgCO2__area_ha,gross_removals__litter_C__MgCO2__area_ha,net_flux__AGC__MgCO2__area_ha,net_flux__BGC__MgCO2__area_ha,net_flux__all_C_pools__CO2_only__MgCO2__area_ha,net_flux__all_C_pools__all_gases__MgCO2e__area_ha,net_flux__deadwood_C__MgCO2__area_ha,net_flux__litter_C__MgCO2__area_ha
0,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,AGO,Angola,Africa,Middle Africa,Africa,Tropical dry forest,Africa-Tropical dry forest,Subtropical/tropical,0,0,0,0,0,0,0,0,0,-670,-194,-1049,-173,-11,-670,-194,-1049,-1049,-173,-11,0,0,0,0,0,0,0,0,0,123,123,123,123,123,123,123,123,123,123,123
1,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,AIA,Anguilla,North America,Caribbean,America South,Tropical dry forest,America South-Tropical dry forest,Subtropical/tropical,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,-1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,ARE,United Arab Emirates,Asia,Western Asia,Asia,Tropical desert,Asia-Tropical desert,Subtropical/tropical,0,0,0,0,0,0,0,0,0,-204,-59,-319,-53,-3,-204,-59,-319,-319,-53,-3,0,0,0,0,0,0,0,0,0,37,37,37,37,37,37,37,37,37,37,37
3,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,ATG,Antigua and Barbuda,North America,Caribbean,America South,Tropical dry forest,America South-Tropical dry forest,Subtropical/tropical,0,0,0,0,0,0,0,0,0,-164,-47,-256,-42,-3,-164,-47,-256,-256,-42,-3,0,0,0,0,0,0,0,0,0,30,30,30,30,30,30,30,30,30,30,30
4,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,AUS,Australia,Oceania,Australia and New Zealand,Asia,Subtropical desert,Asia-Subtropical desert,Subtropical/tropical,0,0,0,0,0,0,0,0,0,-20385,-19569,-45558,-5259,-345,-20385,-19569,-45558,-45558,-5259,-345,0,0,0,0,0,0,0,0,0,683,683,683,683,683,683,683,683,683,683,683


CPU times: user 25 ms, sys: 598 µs, total: 25.6 ms
Wall time: 23.8 ms


In [ ]:
%%time

# Reads SOC zonal stats parquet table
SOC_df_raw = pd.read_parquet(f'{SOC_zonal_stats_folder}{SOC_parquet_name}')

# Renames value column with units
SOC_df_raw.rename(columns={'value': 'zonal_stats_sum'}, inplace=True)

# New column to specify this is mineral soil, as opposed to vegetation or organic soil (for combined LULUCF table).
SOC_df_raw["LULUCF_component"] = "mineral_soil"

print(f"Rows in SOC_df_raw: {len(SOC_df_raw)}")
# SOC_df_aligned.to_csv("/mnt/c/GIS/SOC_raw.csv", index=False)


### Fills in the years for multi-year interval SOC data.
# 2010 is copied to 2006-2010, 2015 to 2011-2015, 2020 to 2016-2020, and 2022 to 2021-2024 (to match end of vegetation timeseries). 
# If a year doesn't have data for a given year and combination of contextual layers, there is nothing to expand and all years in that interval are empty.

series_cols = SOC_df_raw.drop(columns=['flux_Mg_CO2e_yr', 'density__Mg_ha', 'area_ha']).columns.to_list()
print(series_cols)
SOC_df_years_filled_in = fill_in_soil_years(SOC_df_raw, series_cols)
print(f"Rows in SOC_df_years_filled_in: {len(SOC_df_years_filled_in)}")
SOC_df_years_filled_in

# Should be very close to 4.75. If every contextual combination has every year, full year expansion would be 4.75 (2010, 2015 and 2020 intervals expanded 5x years, 2022 expanded 4x years).
# However, some contextual combinations don't have all years (usually because they're so rare and other years have just 1 pixel of that combination),
# so that doesn't get expanded to other years. This results in deviation from the 4.25x expansion. 
print(f"Ratio of rows in unexpanded to expanded tables: {len(SOC_df_years_filled_in)/len(SOC_df_raw)}")

# Drops all years before vegetation data because we don't need those for LULUCF
SOC_df_years_filled_in_post_2016 = SOC_df_years_filled_in[SOC_df_years_filled_in["year"] >= cn.interval_end_years_annual[0]].reset_index(drop=True)
print(f"Rows in SOC_df_years_filled_in_post_2016: {len(SOC_df_years_filled_in_post_2016)}")
SOC_df_years_filled_in_post_2016

SOC_df_years_filled_in_post_2016.to_csv(f'{SOC_zonal_stats_folder}SOC_years_filled_in_post_2016_{today}.csv', index=False, na_rep="NA")

SOC_year_count = len(SOC_df_years_filled_in_post_2016['year'].unique())
print(f"Years in SOC data: {SOC_year_count}")

# QC: Timeseries of SOC change (full and mineral extent) to compare against chunk stats (Mg CO2/yr)
# Some chunk stat values to check against: Density mineral extent 2020 = 350361333214; Net mineral extent 2020=1264590854; gain mineral extent 2020=-5495786224; loss mineral extent 2020=6760376953
layers = [
    "SOC_density__mineral_soil_extent__0-30cm_MgC_ha",
    "SOC_gain__mineral_soil_extent__0-30cm_MgCO2",
    "SOC_loss__mineral_soil_extent__0-30cm_MgCO2",
    "SOC_net__mineral_soil_extent__0-30cm_MgCO2"
]
df_sum = (
    SOC_df_years_filled_in_post_2016[SOC_df_years_filled_in_post_2016["analysis_layer"].isin(layers)]
    .groupby(["analysis_layer", "year"], as_index=False)["flux_Mg_CO2e_yr"]
    .sum()
)
df_sum["flux_Gt_CO2_yr"] = df_sum["flux_Mg_CO2e_yr"] / 1e9 
df_sum

In [ ]:
# # Makes a minimal table of SOC density and change by country and interval 

# df = pd.read_parquet(f'{SOC_zonal_stats_folder}{SOC_parquet_name}')
# df
# df_sum = (
#     df
#     .groupby(["analysis_layer", "year", "adm0", "country_name"], as_index=False)["value"]
#     .sum()
# )
# df_sum.to_csv(f'{SOC_zonal_stats_folder}SOC_by_admin_{today}.csv', index=False, na_rep="NA")

In [ ]:
%%time

# Reads organic soil zonal stats parquet table
org_soil_df_raw = pd.read_csv(f'{org_soil_zonal_stats_folder}{org_soil_csv_name}')
# org_soil_df_raw.columns

# Makes table columns generally match vegetation and SOC
org_soil_df_raw = org_soil_df_raw.drop(columns=['gadm_adm0', 'country'])
org_soil_df_raw.rename(columns={'iso3': 'adm0', 
                                'emissions_MgCO2e': 'flux_Mg_CO2e_yr',   # Erin confirmed in Slack that the values are already Mg CO2e/yr
                                'component': 'analysis_layer',
                                'interval_end': 'year'},
                       inplace=True)
org_soil_df_raw

# # Renames some countries with long names
# org_soil_df_raw["country_name"] = org_soil_df_raw["country_name"].replace({         
#     "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
#     "Russian Federation": "Russia",
#     "Democratic Republic of the Congo": "DR Congo",
#     "United States of America (the)": "USA"
# })

# org_soil_df_raw["climate_domain"] = org_soil_df_raw["climate_domain"].replace({         
#     "tropical": "Subtropical/tropical",
#     "temperate": "Temperate",
#     "boreal": "Boreal",
#     "Unspecified": "Other"
# })

# org_soil_df_raw['country_name'] = org_soil_df_raw[cn.adm0_pattern].map(cn.iso_to_country)
# org_soil_df_raw["continent_ecozone"] = "Unassigned"
# org_soil_df_raw["continent_ecozone"] = org_soil_df_raw["continent_ecozone"].fillna("Unassigned")  
# org_soil_df_raw["country_name"] = org_soil_df_raw["country_name"].fillna("Unassigned")  

# org_soil_df_raw['region_L1'] = org_soil_df_raw[cn.adm0_pattern].map(cn.iso_to_region_UN_geoscheme_L1)
# org_soil_df_raw['region_L2_L3'] = org_soil_df_raw[cn.adm0_pattern].map(cn.iso_to_region_UN_geoscheme_L2_L3)
# org_soil_df_raw['region_L1'] = org_soil_df_raw['region_L1'].fillna("Unassigned")
# org_soil_df_raw['region_L2_L3'] = org_soil_df_raw['region_L2_L3'].fillna("Unassigned")

# # Adds columns to organic soil that are in the vegetation table so the tables can be combined
# # https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c2fe30-7004-8328-9f6d-1c231ba04e20
# target_cols = list(veg_df_raw.columns) + [
#     c for c in org_soil_df_raw.columns if c not in veg_df_raw.columns
# ]
# org_soil_df_aligned = org_soil_df_raw.reindex(columns=target_cols)
# print("veg_df_graphs:", veg_df_raw.columns)
# print("org_soil_df_aligned:", org_soil_df_aligned.columns)

# # Prepends org_soil to rows to clarify their scope
# org_soil_df_aligned["analysis_layer"] = org_soil_df_aligned["analysis_layer"].str.lower()
# org_soil_df_aligned["analysis_layer"] = "organic_soil_" + org_soil_df_aligned["analysis_layer"] + "__all_gases__MgCO2e"

# # New column to specify this is mineral soil, as opposed to vegetation or organic soil (for combined LULUCF table)
# org_soil_df_aligned["LULUCF_component"] = "organic_soil"

# print(f"Rows in df: {len(org_soil_df_aligned)}")
# # org_soil_df_aligned.to_csv("/mnt/c/GIS/org_soil_aligned.csv", index=False)


# ### Fills in the years for multi-year interval SOC data.
# # 2010 is copied to 2006-2010, 2015 to 2011-2015, 2020 to 2016-2020, and 2024 to 2021-2024. 
# # If a year doesn't have data for a given year and combination of contextual layers, there is nothing to expand and all years in that interval are empty. 
# org_soil_df_years_filled_in = fill_in_soil_years(org_soil_df_aligned, series_cols)
# # org_soil_df_years_filled_in.to_csv("/mnt/c/GIS/org_soil_years_filled_in.csv", index=False)
# print(f"Rows in org_soil_df_years_filled_in: {len(org_soil_df_years_filled_in)}")

# # Should be around 4.75. If every contextual combination has every year, full year expansion would be 4.75 (2010, 2015 and 2020 intervals expanded 5x years, 2022 expanded 4x years).
# # However, some contextual combinations don't have all years (usually because they're so rare and other years have just 1 pixel of that combination),
# # so that doesn't get expanded to other years. This results in deviation from the 4.75x expansion. 
# print(f"Ratio of rows in unexpanded to expanded tables: {len(org_soil_df_years_filled_in)/len(org_soil_df_aligned)}")

# # Number of years in organic soil data
# org_soil_year_count = len(org_soil_df_years_filled_in['year'].unique())
# print(f"Years of organic soil data: {org_soil_year_count}")

# # Drops all years before vegetation data because we don't need those for LULUCF
# org_soil_df_years_filled_in_post_2016 = org_soil_df_years_filled_in[org_soil_df_years_filled_in["year"] >= cn.interval_end_years_annual[0]].reset_index(drop=True)

# org_soil_df_years_filled_in_post_2016.to_csv(f"{org_soil_zonal_stats_folder}org_soil_years_filled_in_post_2016_{today}.csv", index=False)
# org_soil_df_years_filled_in_post_2016

In [ ]:
%%time

# Create LULUCF dataframe with net LULUCF fluxes
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c6b94c-b5c0-8332-bd49-807275f66beb

LULUCF = pd.concat([veg_df_outputs_dropped, SOC_df_years_filled_in_post_2016, org_soil_df_years_filled_in_post_2016], ignore_index=True)

# Drops area and density since organic soil table doesn't have either
LULUCF = LULUCF.drop(columns=['area_ha', 'density__Mg_ha'])

df = LULUCF.copy()

components_to_sum = [
    "veg_net_flux__all_C_pools__all_gases__MgCO2e",
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2",
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]

# Keep rows with only the five flux components
df_components = df[df["analysis_layer"].isin(components_to_sum)].copy()

# Contextual columns to group by
# Exclude the layer name and the flux field being summed.
group_cols = [
    c for c in df_components.columns
    if c not in ["analysis_layer", "flux_Mg_CO2e_yr"]
]

# Sum flux across the five components for each unique context combination
df_lulucf_total = (
    df_components
    .groupby(group_cols, dropna=False, as_index=False)
    .agg({
        "flux_Mg_CO2e_yr": "sum"
    })
)

# Add the new layer name
df_lulucf_total["analysis_layer"] = "LULUCF_net_flux__MgCO2e"

# Set a LULUCF label for the component field
df_lulucf_total["LULUCF_component"] = "total_LULUCF"

# Reorder columns to match the existing LULUCF dataframe
df_lulucf_total = df_lulucf_total[df.columns]

# Append to bottom of original table
LULUCF = pd.concat([df, df_lulucf_total], ignore_index=True)
print(f"Rows in df: {len(LULUCF)}")
LULUCF.head()

### Dataframes with less necessary analysis layers removed

In [ ]:
%%time

# Drops analysis layers from SOC dataframe so it can be graphed more easily
SOC_layers_to_drop = [
    "SOC_density__full_extent__0-30cm_MgC",
    "SOC_density__mineral_soil_extent__0-30cm_MgC",
    "SOC_change__full_extent__0-30cm_MgCO2"
]

SOC_df_outputs_dropped = SOC_df_years_filled_in_post_2016[~SOC_df_years_filled_in_post_2016["analysis_layer"].isin(SOC_layers_to_drop)].reset_index(drop=True)
print(f"Rows in SOC_df_outputs_dropped: {len(SOC_df_outputs_dropped)}")
# SOC_df_outputs_dropped.to_csv("/mnt/c/GIS/SOC_outputs_dropped.csv", index=False)
# SOC_df_outputs_dropped.head()

In [ ]:
# Not currently dropping any output layers from organic soil

org_soil_outputs_dropped = org_soil_df_years_filled_in_post_2016.copy()

In [ ]:
%%time
# Drops analysis layers from LULUCF dataframe so it can be graphed more easily.
# Already dropped the veg layers from the veg dataframe earlier because it was crashing Jupyter, so they are not actually necessary here. 

LULUCF_layers_to_drop = [
    f"veg_carbon_density__non_soil__MgC_ha",
    f"veg_{cn.agc_gross_emis_pattern}",
    f"veg_{cn.bgc_gross_emis_pattern}",
    f"veg_{cn.deadwood_c_gross_emis_pattern}",
    f"veg_{cn.litter_c_gross_emis_pattern}",
    f"veg_{cn.net_flux_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.agc_gross_removals_pattern}",
    f"veg_{cn.bgc_gross_removals_pattern}",
    f"veg_{cn.deadwood_c_gross_removals_pattern}",
    f"veg_{cn.litter_c_gross_removals_pattern}",
    f"veg_{cn.net_flux_agc_pattern}",
    f"veg_{cn.net_flux_bgc_pattern}",
    f"veg_{cn.net_flux_deadwood_c_pattern}",
    f"veg_{cn.net_flux_litter_c_pattern}",
    f"veg_{cn.ch4_gross_emis_pattern}",
    f"veg_{cn.n2o_gross_emis_pattern}",
    "SOC_density__full_extent__0-30cm_MgC",
    "SOC_density__mineral_soil_extent__0-30cm_MgC",
    "SOC_change__full_extent__0-30cm_MgCO2"
]

LULUCF_outputs_dropped = LULUCF[~LULUCF["analysis_layer"].isin(LULUCF_layers_to_drop)].reset_index(drop=True)

print(f"Rows in df: {len(LULUCF_outputs_dropped)}")
LULUCF_outputs_dropped.head()

In [ ]:
# QC for LULUCF table-- makes sure that all columns have expected lists of values

print(f"Columns in LULUCF table are: {LULUCF_outputs_dropped.columns}")
for column in LULUCF_outputs_dropped.columns:
    print(f"{column}: {LULUCF_outputs_dropped[column].unique()}")

In [ ]:
# %%time

# # Creates wide format table of LULUCF with more detailed outputs (analysis_layers) dropped

# LULUCF_outputs_dropped_wide = create_wide_df(LULUCF_outputs_dropped, 'flux_Mg_CO2e_yr')
# print(f"Rows in df: {len(LULUCF_outputs_dropped_wide)}: {uu.timestr()}")
# LULUCF_outputs_dropped_wide.to_csv(f'{LULUCF_zonal_stats_folder}/LULUCF_outputs_dropped_wide_{today}.csv', index=False, na_rep="NA")
# LULUCF_outputs_dropped_wide.head()

### Dataframes with less necessary contextual layers removed, then reaggregated for export to csv

In [ ]:
%%time

# Creates vegetation dataframe to export to Excel for manual graphing of timeseries..
# per https://www.google.com/search?q=how+to+reaggregate+a+dataframe+by+a+few+columns&rlz=1C1GCEA_enUS1147US1147&oq=how+to+reaggregate+a+dataframe+by+a+few+columns&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIJCAEQIRgKGKABMgkIAhAhGAoYoAEyCQgDECEYChigATIJCAQQIRgKGKAB0gEIODkwM2owajeoAgCwAgA&sourceid=chrome&ie=UTF-8

veg_df_graphs_agg_annual = (
    veg_df_outputs_dropped
      .groupby(export_csv_group_cols, dropna=False, as_index=False)
      .agg({'flux_Mg_CO2e_yr': 'sum', 'area_ha': 'sum'})  # All other columns are ignored. If specific columns aren't identified for summing, then things like WDPA and tile_id also get (meaninglessly) summed
)
print(f"Rows in veg timeseries df: {len(veg_df_graphs_agg_annual)}")
# veg_df_graphs_agg_annual.to_csv(f'{veg_zonal_stats_folder}veg_model_zonal_stats_aggregated_timeseries_20260401.csv', index=False)
veg_df_graphs_agg_annual

In [ ]:
%%time

# Creates SOC dataframe to export to Excel for manual graphing of timeseries..
# per https://www.google.com/search?q=how+to+reaggregate+a+dataframe+by+a+few+columns&rlz=1C1GCEA_enUS1147US1147&oq=how+to+reaggregate+a+dataframe+by+a+few+columns&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIJCAEQIRgKGKABMgkIAhAhGAoYoAEyCQgDECEYChigATIJCAQQIRgKGKAB0gEIODkwM2owajeoAgCwAgA&sourceid=chrome&ie=UTF-8

SOC_df_graphs_agg_annual = (
    SOC_df_outputs_dropped
      .groupby(export_csv_group_cols, dropna=False, as_index=False)
      .agg({'flux_Mg_CO2e_yr': 'sum', 'area_ha': 'sum'})  # All other columns are ignored. If specific columns aren't identified for summing, then things like WDPA and tile_id also get (meaninglessly) summed
)
print(f"Rows in SOC timeseries df: {len(SOC_df_graphs_agg_annual)}")
# SOC_df_graphs_agg_annual.to_csv(f'{SOC_zonal_stats_folder}SOC_zonal_stats_aggregated_timeseries_20260401.csv', index=False, na_rep="NA")
SOC_df_graphs_agg_annual

In [ ]:
%%time

# Creates organic soil dataframe to export to Excel for manual graphing of timeseries..
# per https://www.google.com/search?q=how+to+reaggregate+a+dataframe+by+a+few+columns&rlz=1C1GCEA_enUS1147US1147&oq=how+to+reaggregate+a+dataframe+by+a+few+columns&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIJCAEQIRgKGKABMgkIAhAhGAoYoAEyCQgDECEYChigATIJCAQQIRgKGKAB0gEIODkwM2owajeoAgCwAgA&sourceid=chrome&ie=UTF-8

org_soil_df_graphs_agg_annual = (
    org_soil_outputs_dropped
      .groupby(export_csv_group_cols, dropna=False, as_index=False)
      .agg({'flux_Mg_CO2e_yr': 'sum', 'area_ha': 'sum'})  # All other columns are ignored. If specific columns aren't identified for summing, then things like WDPA and tile_id also get (meaninglessly) summed
)
print(f"Rows in organic soil timeseries df: {len(org_soil_df_graphs_agg_annual)}")
# org_soil_df_graphs_agg_annual.to_csv(f'{org_soil_zonal_stats_folder}organic_soil_zonal_stats_aggregated_timeseries_{today}.csv', index=False, na_rep="NA")
org_soil_df_graphs_agg_annual

In [ ]:
%%time

# Creates LULUCF dataframe to export to Excel for manual graphing of annual averages (drops other contextual layers, too).
# per https://www.google.com/search?q=how+to+reaggregate+a+dataframe+by+a+few+columns&rlz=1C1GCEA_enUS1147US1147&oq=how+to+reaggregate+a+dataframe+by+a+few+columns&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIJCAEQIRgKGKABMgkIAhAhGAoYoAEyCQgDECEYChigATIJCAQQIRgKGKAB0gEIODkwM2owajeoAgCwAgA&sourceid=chrome&ie=UTF-8

# Grouping columns for summary table
summary_table_group_columns = ["analysis_layer", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "LULUCF_component"]

# The layers needed for the final summary table
layers_to_keep = [
    f"veg_{cn.ch4_gross_emis_pattern}",
    f"veg_{cn.n2o_gross_emis_pattern}",
    f"veg_{cn.gross_emis_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}",
    f"veg_{cn.gross_removals_all_C_pools_pattern}",
    f"veg_{cn.net_flux_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2"
]

LULUCF_summary_table = LULUCF[LULUCF['analysis_layer'].isin(layers_to_keep)].reset_index(drop=True)

# Total fluxes over the entire 
LULUCF_summary_table = (
    LULUCF_summary_table
      .groupby(summary_table_group_columns, dropna=False, as_index=False)
      .agg({'flux_Mg_CO2e_yr': 'sum'})  # All other columns are ignored. If specific columns aren't identified for summing, then things like WDPA and tile_id also get (meaninglessly) summed
)

# Converts total fluxes from entire model period to annual averages
# TODO The separate treatment for vegetation and soil is kind of a hack. Need to revisit. From Claude. 
divisor_map = {
    f"veg_{cn.ch4_gross_emis_pattern}": len(cn.interval_end_years_annual),
    f"veg_{cn.n2o_gross_emis_pattern}": len(cn.interval_end_years_annual),
    f"veg_{cn.gross_emis_all_C_pools_CO2_only_pattern}": len(cn.interval_end_years_annual),
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}": len(cn.interval_end_years_annual),
    f"veg_{cn.gross_removals_all_C_pools_pattern}": len(cn.interval_end_years_annual),
    f"veg_{cn.net_flux_all_C_pools_CO2_only_pattern}": len(cn.interval_end_years_annual),
    f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}": len(cn.interval_end_years_annual),
    "organic_soil_drainage__all_gases__MgCO2e": org_soil_year_count,
    "organic_soil_extraction__all_gases__MgCO2e": org_soil_year_count,
    "organic_soil_fire__all_gases__MgCO2e": org_soil_year_count,
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2": SOC_year_count,
}

LULUCF_summary_table["flux_Mg_CO2e_yr"] = (
    LULUCF_summary_table["flux_Mg_CO2e_yr"]
    / LULUCF_summary_table["analysis_layer"].map(divisor_map)
)

# QC: These pairs of values should match
print(LULUCF[LULUCF['analysis_layer']=='organic_soil_drainage__all_gases__MgCO2e']['flux_Mg_CO2e_yr'].sum()/org_soil_year_count)
print(LULUCF_summary_table[LULUCF_summary_table['analysis_layer']=='organic_soil_drainage__all_gases__MgCO2e']['flux_Mg_CO2e_yr'].sum())
print(LULUCF[LULUCF['analysis_layer']=='SOC_change__mineral_soil_extent__0-30cm_MgCO2']['flux_Mg_CO2e_yr'].sum()/SOC_year_count)
print(LULUCF_summary_table[LULUCF_summary_table['analysis_layer']=='SOC_change__mineral_soil_extent__0-30cm_MgCO2']['flux_Mg_CO2e_yr'].sum())

print(LULUCF_summary_table['analysis_layer'].value_counts())

print(f"Rows in annual average LULUCF df: {len(LULUCF_summary_table)}")
# LULUCF_summary_table.to_csv(f'{LULUCF_zonal_stats_folder}/LULUCF_zonal_stats_for_summary_table_{today}.csv', index=False, na_rep="NA")
LULUCF_summary_table

### Dataset transformation QC

In [ ]:
# Confirms that global totals for vegetation net flux in the raw data and final data match each other for a susbet of years

test_years = [2015, 2016, 2019, 2020, 2023, 2024]  # Not supposed to be all years
for year in test_years:
    raw = veg_df_raw.loc[(veg_df_raw["year"] == year) & (veg_df_raw["analysis_layer"] == "veg_net_flux__all_C_pools__all_gases__MgCO2e"), "flux_Mg_CO2e_yr"].sum()
    final = LULUCF_outputs_dropped.loc[(LULUCF_outputs_dropped["year"] == year) & (LULUCF_outputs_dropped["analysis_layer"] == "veg_net_flux__all_C_pools__all_gases__MgCO2e"), "flux_Mg_CO2e_yr"].sum()

    print(f"Year: {year}")
    print(f"  Raw:   {raw}")
    print(f"  Final: {final}")
    print(f"  Percent difference: {(raw-final)/raw*100}%")

In [ ]:
# Confirms that global totals for SOC timeseries in the raw data and final data match each other for representative years

test_years = [2010, 2011, 2015, 2016, 2020, 2022, 2024]  # Not supposed to be all years
for year in test_years:
    raw = SOC_df_raw.loc[(SOC_df_raw["year"] == year) & (SOC_df_raw["analysis_layer"] == "SOC_change__mineral_soil_extent__0-30cm_MgCO2"), "flux_Mg_CO2e_yr"].sum()
    final = LULUCF_outputs_dropped.loc[(LULUCF_outputs_dropped["year"] == year) & (LULUCF_outputs_dropped["analysis_layer"] == "SOC_change__mineral_soil_extent__0-30cm_MgCO2"), "flux_Mg_CO2e_yr"].sum()

    print(f"Year: {year}")
    print(f"  Raw:   {raw}")
    print(f"  Final: {final}")
    print(f"  Percent difference: {(raw-final)/raw*100}%")

In [ ]:
# Confirms that global totals for organic soil timeseries in the raw data and final data match each other for representative years

test_years = [2010, 2011, 2015, 2016, 2020, 2022, 2024]  # Not supposed to be all years
for year in test_years:
    raw = org_soil_df_raw.loc[(org_soil_df_raw["year"] == year) & (org_soil_df_raw["analysis_layer"] == "Drainage"), "flux_Mg_CO2e_yr"].sum()
    final = LULUCF_outputs_dropped.loc[(LULUCF_outputs_dropped["year"] == year) & (LULUCF_outputs_dropped["analysis_layer"] == "organic_soil_drainage__all_gases__MgCO2e"), "flux_Mg_CO2e_yr"].sum()

    print(f"Year: {year}")
    print(f"  Raw:   {raw}")
    print(f"  Final: {final}")
    print(f"  Percent difference: {(raw-final)/raw*100}%")

### Figure creation

In [ ]:
# General code for figures

years = len(cn.interval_end_years_annual)

gross_emis_all_gases_legend = "Gross emissions (all gases)"
gross_emis_CO2_legend = "Gross emissions (CO₂ only)"
gross_emis_non_CO2_legend = "Gross emissions (non-CO₂)"
net_flux_all_gases_legend = "Net flux (all gases)"
gross_removals_legend = "Gross removals"

broad_class_dict = {
    "tree": "Tall vegetation",
    "short_veg": "Short vegetation",
    "crop": "Cropland"
}

detailed_class_dict = {
    "tree_tree_undisturbed": "Undisturbed tall veg",
    "tree_tree_disturbed": "Partially disturbed tall veg", 
    "tree_tree_disturbed_fire_ony": "Tall veg with fire but not height reduction", 
    "tree_loss": "Tall veg loss",
    "tree_gain": "Tall veg gain",
    "short_veg_short_veg_undisturbed": "Stable short veg",
    "short_veg_gain": "Short veg gain",
    "short_veg_loss": "Short veg loss", 
    "crop_crop_undisturbed": "Stable cropland",
    "crop_gain": "Cropland gain",
    "crop_loss": "Cropland loss"
}

tall_veg_type_dict = {
    "mangrove": "Mangrove",
    "oil_palm": "Oil palm",
    "non_oil_palm_planted_trees": "Non-oil palm planted trees",
    "natural_tree_cover": "Natural tree cover",
    "trees_in_other_land_covers": "Trees in other land covers",
    "non_tall_vegetation": "Non-tall vegetation"
}

primary_forest_dict = {
    0: "Not primary forest",
    1: "Primary forest"
}

tall_veg_change_type_classes = [
    "Tall veg loss",
    "Tall veg gain",
    "Partially disturbed tall veg",
    "Undisturbed (except for fires) tall veg"
]

# Aggregates by analysis layer
def aggregate_layer(df, pattern, context_to_analyze):
    d = df[df["analysis_layer"] == pattern]
    d = (
        d.groupby([context_to_analyze, "year"], as_index=False)
         .agg({"value": "sum"})
    )
    d["value_Gt"] = d["value"] / 1e9
    return d

sns.set_theme(style="white", context="talk")

In [ ]:
### Global LULUCF components with gross as bars and net veg and LULUCF as lines (single panel)
### Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c6d457-96b4-8328-9fca-ff50f8273233
### Colors consistent with LULUCF maps from Claude (session 'LULUCF graph color consistency with maps')

df_fig = LULUCF_outputs_dropped.copy()

## Various lists and dictionaries needed to create the figure

# Drop unwanted vegetation-only emissions layers
layers_to_drop = [
    f"veg_{cn.gross_emis_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.gross_emis_all_C_pools_non_CO2_only_pattern}",
]

# Net flux layers to draw as lines instead of bars
net_flux_layers = [
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
    "LULUCF_net_flux__MgCO2e",
]

# Organic soil layers to combine for graphing
organic_soil_layers = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]

organic_soil_combined_label = "organic_soil__all_gases__MgCO2e"

desired_bar_order = [
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2",
    "organic_soil__all_gases__MgCO2e",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}",
    f"veg_{cn.gross_removals_all_C_pools_pattern}",
]

desired_line_order = [
    "LULUCF_net_flux__MgCO2e",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
]

color_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2": "#8c510a",        # emissions idx 8 (medium-dark brown)
    "organic_soil__all_gases__MgCO2e":               "#543005",         # emissions idx 9 (darkest brown)
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}": "#dfc27d",   # emissions idx 6 (medium tan)
    f"veg_{cn.gross_removals_all_C_pools_pattern}":  "#35978f",         # removals  idx 2 (medium teal)
    "LULUCF_net_flux__MgCO2e":                       "#000000",         # black line
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}": "#666633",  # green
}

label_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2":
        "Mineral soil carbon change",
    "organic_soil__all_gases__MgCO2e":
        "Organic soil emissions",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}":
        "Vegetation gross emissions",
    f"veg_{cn.gross_removals_all_C_pools_pattern}":
        "Vegetation gross removals",
    "LULUCF_net_flux__MgCO2e":
        "LULUCF net flux",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}":
    #     "Vegetation net flux",
}


## Creates the figure

df_fig = df_fig[~df_fig["analysis_layer"].isin(layers_to_drop)].copy()

# Combine the three organic soil layers into one label
df_fig["analysis_layer"] = df_fig["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)

# Sum by year and analysis layer
df_ts = (
    df_fig
    .groupby(["year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)

# Convert to Gt CO2e/yr
df_ts["flux_Gt_CO2e_yr"] = df_ts["flux_Mg_CO2e_yr"] / 1e9

# Pivot to wide form
df_wide = (
    df_ts
    .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
    .fillna(0)
)

# Apply ordering (keep only columns that exist)
bar_cols = [c for c in desired_bar_order if c in df_wide.columns]
line_cols = [c for c in desired_line_order if c in df_wide.columns]

df_bars = df_wide[bar_cols]
df_lines = df_wide[line_cols]

# Reverse bar order for plotting (top to bottom)
df_bars = df_bars[bar_cols[::-1]]

# Split positive and negative bars so they stack correctly around zero
df_pos = df_bars.clip(lower=0)
df_neg = df_bars.clip(upper=0)

fig, ax = plt.subplots(figsize=(28, 8))

# Positive stacked bars
df_pos.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    width=0.8,
    color=[color_map[c] for c in df_pos.columns]
)

# Negative stacked bars
df_neg.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    width=0.8,
    legend=False,
    color=[color_map[c] for c in df_neg.columns]
)

# Lines
x = np.arange(len(df_wide.index))
for col in line_cols:
    ax.plot(
        x,
        df_lines[col].values,
        linewidth=2.5,
        label=col,
        color=color_map.get(col, "black"),
        zorder=5
    )

# Formatting
ax.axhline(0, color="black", linewidth=1)
ax.grid(visible=True, axis="y")
ax.set_axisbelow(True)
ax.yaxis.set_major_locator(MultipleLocator(5))

ax.set_xlabel(None)
ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
# ax.set_title("Global annual LULUCF fluxes", pad=20)

ax.set_xticks(x[::2])
ax.set_xticklabels(df_wide.index.astype(int)[::2], rotation=0)

# Build legend in desired order
handles, labels = ax.get_legend_handles_labels()
label_to_handle = dict(zip(labels, handles))

ordered_labels = (
    desired_bar_order +
    desired_line_order
)

ordered_labels = [l for l in ordered_labels if l in label_to_handle]

# Map labels to readable names
pretty_labels = [label_map.get(l, l) for l in ordered_labels]

ax.legend(
    [label_to_handle[l] for l in ordered_labels],
    pretty_labels,
    title="",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 0.80, 1])
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_global_timeseries_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# df_wide

In [ ]:
### Global LULUCF components with gross as bars and net veg and LULUCF as lines, panel by climate domain (three panels)
### Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c6d457-96b4-8328-9fca-ff50f8273233
### Colors consistent with LULUCF maps from Claude (session 'LULUCF graph color consistency with maps')

df_fig = LULUCF_outputs_dropped.copy()

## Various lists and dictionaries needed to create the figure

# Drop unwanted vegetation-only emissions layers
layers_to_drop = [
    f"veg_{cn.gross_emis_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.gross_emis_all_C_pools_non_CO2_only_pattern}",
]

# Net flux layers to draw as lines instead of bars
net_flux_layers = [
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
    "LULUCF_net_flux__MgCO2e",
]

# Organic soil layers to combine for graphing
organic_soil_layers = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]

organic_soil_combined_label = "organic_soil__all_gases__MgCO2e"

desired_bar_order = [
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2",
    "organic_soil__all_gases__MgCO2e",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}",
    f"veg_{cn.gross_removals_all_C_pools_pattern}",
]

desired_line_order = [
    "LULUCF_net_flux__MgCO2e",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
]

color_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2": "#8c510a",        # emissions idx 8 (medium-dark brown)
    "organic_soil__all_gases__MgCO2e":               "#543005",         # emissions idx 9 (darkest brown)
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}": "#dfc27d",   # emissions idx 6 (medium tan)
    f"veg_{cn.gross_removals_all_C_pools_pattern}":  "#35978f",         # removals  idx 2 (medium teal)
    "LULUCF_net_flux__MgCO2e":                       "#000000",         # black line
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}": "#666633",  # green
}

label_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2":
        "Mineral soil carbon change",
    "organic_soil__all_gases__MgCO2e":
        "Organic soil emissions",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}":
        "Vegetation gross emissions",
    f"veg_{cn.gross_removals_all_C_pools_pattern}":
        "Vegetation gross removals",
    "LULUCF_net_flux__MgCO2e":
        "LULUCF net flux",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}":
    #     "Vegetation net flux",
}

domain_order = [
    "Subtropical/tropical",
    "Temperate",
    "Boreal",
]

domain_title_map = {
    "Subtropical/tropical": "Subtropical/tropical",
    "Temperate": "Temperate",
    "Boreal": "Boreal",
}


## Creates the figure

df_fig = df_fig[~df_fig["analysis_layer"].isin(layers_to_drop)].copy()

# Keep only requested climate domains
df_fig = df_fig[df_fig["climate_domain"].isin(domain_order)].copy()

# Combine the three organic soil layers into one label
df_fig["analysis_layer"] = df_fig["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)

# Sum by climate domain, year, and analysis layer
df_ts = (
    df_fig
    .groupby(["climate_domain", "year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)

# Convert to Gt CO2e/yr
df_ts["flux_Gt_CO2e_yr"] = df_ts["flux_Mg_CO2e_yr"] / 1e9

# Compute shared y-limits across all panels
all_panel_values = []

for domain in domain_order:
    df_domain = df_ts[df_ts["climate_domain"] == domain]

    if df_domain.empty:
        continue

    df_wide = (
        df_domain
        .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
        .fillna(0)
    )

    bar_cols = [c for c in desired_bar_order if c in df_wide.columns]
    line_cols = [c for c in desired_line_order if c in df_wide.columns]

    if bar_cols:
        df_bars = df_wide[bar_cols]
        all_panel_values.extend(df_bars.to_numpy().ravel())

    if line_cols:
        df_lines = df_wide[line_cols]
        all_panel_values.extend(df_lines.to_numpy().ravel())

y_abs_max = max(abs(np.nanmin(all_panel_values)), abs(np.nanmax(all_panel_values)))
y_pad = y_abs_max * 0.08
ymin, ymax = -y_abs_max - y_pad, y_abs_max + y_pad


## Plot

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)

legend_handles = None
legend_labels = None

for ax, domain in zip(axes, domain_order):
    df_domain = df_ts[df_ts["climate_domain"] == domain]

    if df_domain.empty:
        ax.set_visible(False)
        continue

    df_wide = (
        df_domain
        .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
        .fillna(0)
    )

    # Apply ordering
    bar_cols = [c for c in desired_bar_order if c in df_wide.columns]
    line_cols = [c for c in desired_line_order if c in df_wide.columns]

    df_bars = df_wide[bar_cols]
    df_lines = df_wide[line_cols]

    # Reverse for plotting so desired_bar_order reads top-to-bottom in legend
    df_bars = df_bars[bar_cols[::-1]]

    # Split positive and negative bars
    df_pos = df_bars.clip(lower=0)
    df_neg = df_bars.clip(upper=0)

    # Plot bars
    df_pos.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.8,
        legend=False,
        color=[color_map[c] for c in df_pos.columns]
    )

    df_neg.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.8,
        legend=False,
        color=[color_map[c] for c in df_neg.columns]
    )

    # Plot lines
    x = np.arange(len(df_wide.index))
    for col in line_cols:
        ax.plot(
            x,
            df_lines[col].values,
            linewidth=2.5,
            label=col,
            color=color_map.get(col, "black"),
            zorder=5
        )

    # Formatting
    ax.axhline(0, color="black", linewidth=1)
    ax.set_axisbelow(True)
    ax.grid(visible=True, axis="y")
    ax.set_ylim(ymin, ymax)

    ax.set_title(domain_title_map.get(domain, domain), pad=12)
    ax.set_xlabel(None)

    ax.set_xticks(x[::2])
    ax.set_xticklabels(df_wide.index.astype(int)[::2], rotation=0)

    if ax is axes[0]:
        ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
    else:
        ax.set_ylabel(None)

    # Save handles once for shared legend
    if legend_handles is None:
        handles, labels = ax.get_legend_handles_labels()
        label_to_handle = dict(zip(labels, handles))
        ordered_labels = [l for l in (desired_bar_order + desired_line_order) if l in label_to_handle]
        legend_handles = [label_to_handle[l] for l in ordered_labels]
        legend_labels = [label_map.get(l, l) for l in ordered_labels]

# # Shared title
# fig.suptitle("Global annual LULUCF fluxes by climate domain", y=0.98)

# Shared legend below figure
legend = fig.legend(
    legend_handles,
    legend_labels,
    title="",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=1,
    frameon=False
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.05)  # To reduce space between graphs and legend. Smaller value is smaller gap. 
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_climate_domain_timeseries_with_legend_{today}.jpg', dpi=300, bbox_inches='tight')
legend.set_visible(False)
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_climate_domain_no_legend_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

print(f"Table for {domain} (to compare against Excel spreadsheet):")
df_wide

In [ ]:
### Global flux timeseries (panel a) and climate-domain flux timeseries (panel b) for journal article
### Per Claude session 'LULUCF graph color consistency with maps'
### Claude said it was better to recreate the maps rather than trying to composite the global and climate domain images created above

# ── Figure layout ─────────────────────────────────────────────────────────────
# Panel a:  left=0.07 → right=0.70  (63% of figure width)
# Legend:   right of panel a, ending at ~0.92
# B panels: left=0.07 → right=0.92  (85% of figure width = same as a + legend)

fig = plt.figure(figsize=(20, 13))

gs_a = GridSpec(1, 1, figure=fig,
                left=0.07, right=0.70, top=0.93, bottom=0.52)
ax_a = fig.add_subplot(gs_a[0, 0])

gs_b = GridSpec(1, 3, figure=fig,
                left=0.07, right=0.92, top=0.45, bottom=0.08,
                wspace=0.25)
ax_b0 = fig.add_subplot(gs_b[0, 0])
ax_b1 = fig.add_subplot(gs_b[0, 1], sharey=ax_b0)
ax_b2 = fig.add_subplot(gs_b[0, 2], sharey=ax_b0)
axes_b = [ax_b0, ax_b1, ax_b2]

# ── Panel a: global ───────────────────────────────────────────────────────────

df_fig_a = LULUCF_outputs_dropped.copy()
df_fig_a = df_fig_a[~df_fig_a["analysis_layer"].isin(layers_to_drop)].copy()
df_fig_a["analysis_layer"] = df_fig_a["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)
df_ts_a = (
    df_fig_a
    .groupby(["year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_ts_a["flux_Gt_CO2e_yr"] = df_ts_a["flux_Mg_CO2e_yr"] / 1e9
df_wide_a = (
    df_ts_a
    .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
    .fillna(0)
)

bar_cols_a  = [c for c in desired_bar_order  if c in df_wide_a.columns]
line_cols_a = [c for c in desired_line_order if c in df_wide_a.columns]
df_bars_a   = df_wide_a[bar_cols_a[::-1]]

df_bars_a.clip(lower=0).plot(kind="bar", stacked=True, ax=ax_a, width=0.8, legend=False,
                              color=[color_map[c] for c in df_bars_a.columns])
df_bars_a.clip(upper=0).plot(kind="bar", stacked=True, ax=ax_a, width=0.8, legend=False,
                              color=[color_map[c] for c in df_bars_a.columns])

x_a = np.arange(len(df_wide_a.index))
for col in line_cols_a:
    ax_a.plot(x_a, df_wide_a[col].values, linewidth=2.5,
              color=color_map.get(col, "black"), zorder=5)

ax_a.axhline(0, color="black", linewidth=1)
ax_a.grid(visible=True, axis="y")
ax_a.set_axisbelow(True)
ax_a.yaxis.set_major_locator(MultipleLocator(5))
ax_a.set_xlabel(None)
ax_a.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
ax_a.set_xticks(x_a[::2])
ax_a.set_xticklabels(df_wide_a.index.astype(int)[::2], rotation=0)

# ── Panel b: climate domains ──────────────────────────────────────────────────

df_fig_b = LULUCF_outputs_dropped.copy()
df_fig_b = df_fig_b[~df_fig_b["analysis_layer"].isin(layers_to_drop)].copy()
df_fig_b = df_fig_b[df_fig_b["climate_domain"].isin(domain_order)].copy()
df_fig_b["analysis_layer"] = df_fig_b["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)
df_ts_b = (
    df_fig_b
    .groupby(["climate_domain", "year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_ts_b["flux_Gt_CO2e_yr"] = df_ts_b["flux_Mg_CO2e_yr"] / 1e9

# Shared y-limits across b panels
all_b_vals = []
for domain in domain_order:
    df_d = df_ts_b[df_ts_b["climate_domain"] == domain]
    if not df_d.empty:
        df_w = df_d.pivot(index="year", columns="analysis_layer",
                          values="flux_Gt_CO2e_yr").fillna(0)
        all_b_vals.extend(df_w.to_numpy().ravel())

y_abs_max_b = max(abs(np.nanmin(all_b_vals)), abs(np.nanmax(all_b_vals)))
y_pad_b = y_abs_max_b * 0.08

for ax, domain in zip(axes_b, domain_order):
    df_domain = df_ts_b[df_ts_b["climate_domain"] == domain]
    if df_domain.empty:
        ax.set_visible(False)
        continue

    df_wide = (
        df_domain
        .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
        .fillna(0)
    )
    bar_cols  = [c for c in desired_bar_order  if c in df_wide.columns]
    line_cols = [c for c in desired_line_order if c in df_wide.columns]
    df_bars   = df_wide[bar_cols[::-1]]

    df_bars.clip(lower=0).plot(kind="bar", stacked=True, ax=ax, width=0.8, legend=False,
                                color=[color_map[c] for c in df_bars.columns])
    df_bars.clip(upper=0).plot(kind="bar", stacked=True, ax=ax, width=0.8, legend=False,
                                color=[color_map[c] for c in df_bars.columns])

    x = np.arange(len(df_wide.index))
    for col in line_cols:
        ax.plot(x, df_wide[col].values, linewidth=2.5,
                color=color_map.get(col, "black"), zorder=5)

    ax.axhline(0, color="black", linewidth=1)
    ax.set_axisbelow(True)
    ax.grid(visible=True, axis="y")
    ax.set_ylim(-y_abs_max_b - y_pad_b, y_abs_max_b + y_pad_b)
    ax.set_title(domain_title_map.get(domain, domain), pad=12)
    ax.set_xlabel(None)
    ax.set_xticks(x[::2])
    ax.set_xticklabels(df_wide.index.astype(int)[::2], rotation=0)
    if ax is axes_b[0]:
        ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")

# ── Panel labels ──────────────────────────────────────────────────────────────

ax_a.text(-0.02, 1.03, "a", transform=ax_a.transAxes,
          fontsize=14, fontweight="bold", va="bottom")
axes_b[0].text(-0.10, 1.03, "b", transform=axes_b[0].transAxes,
               fontsize=14, fontweight="bold", va="bottom")

# ── Legend ────────────────────────────────────────────────────────────────────

legend_items = [
    Patch(facecolor=color_map["SOC_change__mineral_soil_extent__0-30cm_MgCO2"],
          label=label_map["SOC_change__mineral_soil_extent__0-30cm_MgCO2"]),
    Patch(facecolor=color_map["organic_soil__all_gases__MgCO2e"],
          label=label_map["organic_soil__all_gases__MgCO2e"]),
    Patch(facecolor=color_map[f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"],
          label=label_map[f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"]),
    Patch(facecolor=color_map[f"veg_{cn.gross_removals_all_C_pools_pattern}"],
          label=label_map[f"veg_{cn.gross_removals_all_C_pools_pattern}"]),
    Line2D([0], [0], color=color_map["LULUCF_net_flux__MgCO2e"], linewidth=2.5,
           label=label_map["LULUCF_net_flux__MgCO2e"]),
]

# Anchor legend just right of panel a, vertically centred with it
fig.canvas.draw()
ax_a_pos  = ax_a.get_position()
legend_x  = ax_a_pos.x1 + 0.01
legend_y  = ax_a_pos.y0 + ax_a_pos.height / 2

legend = fig.legend(
    handles=legend_items,
    loc="center left",
    bbox_to_anchor=(legend_x, legend_y),
    frameon=False,
)

# ── Save ──────────────────────────────────────────────────────────────────────

# fig.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_global_and_climate_domain_with_legend_{today}.jpg', dpi=300, bbox_inches='tight')

In [ ]:
### LULUCF waterfall diagram
### Made with Claude ('Create LULUCF waterfall chart')

# ── 0. Aggregate raw data into summary rows ──────────────────────────────────
raw = LULUCF_outputs_dropped[
    ~LULUCF_outputs_dropped["analysis_layer"].str.contains("gross_emissions|gross_removals")
].copy()

n_years = raw["year"].nunique()

def annual_Gt(df, mask):
    """Sum all pixel-years matching mask, divide by n_years → annual average in Gt."""
    return df.loc[mask, "flux_Mg_CO2e_yr"].sum() / n_years / 1e9

rows_summary = []
def add_row(section, subsection, category, val):
    rows_summary.append({
        "Section": section, "Subsection": subsection,
        "Category": category, "All_gases_Gt_CO2e_yr": val,
    })

veg  = raw["LULUCF_component"] == "vegetation"
tree = raw["land_state_broad_class"] == "tree"
dlc  = raw["land_state_detailed_class"]

add_row("Vegetation", "Tall vegetation", "New tree cover",
        annual_Gt(raw, veg & tree & (dlc == "tree_gain")))
add_row("Vegetation", "Tall vegetation", "Undisturbed tree cover (including fires)",
        annual_Gt(raw, veg & tree & (dlc == "tree_tree_undisturbed")))
add_row("Vegetation", "Tall vegetation", "Partially disturbed tree cover",
        annual_Gt(raw, veg & tree & (dlc.isin(["tree_tree_disturbed", "tree_tree_disturbed_fire_only"]))))
add_row("Vegetation", "Tall vegetation", "Tree cover loss",
        annual_Gt(raw, veg & tree & (dlc == "tree_loss")))
add_row("Vegetation", "Tall vegetation", "Total",
        annual_Gt(raw, veg & tree))
add_row("Vegetation", "Non-tall vegetation", "All classes",
        annual_Gt(raw, veg & (raw["tall_veg_type"] == "non_tall_vegetation")))
add_row("Vegetation", "All vegetation", "Total",
        annual_Gt(raw, veg))
add_row("Soil", "Organic soil", "Fire",
        annual_Gt(raw, raw["analysis_layer"].str.contains("organic_soil_fire")))
add_row("Soil", "Organic soil", "Extraction",
        annual_Gt(raw, raw["analysis_layer"].str.contains("organic_soil_extraction")))
add_row("Soil", "Organic soil", "Drainage",
        annual_Gt(raw, raw["analysis_layer"].str.contains("organic_soil_drainage")))
add_row("Soil", "Organic soil", "Total",
        annual_Gt(raw, raw["LULUCF_component"] == "organic_soil"))
add_row("Soil", "Mineral soil", "All classes",
        annual_Gt(raw, raw["LULUCF_component"] == "mineral_soil"))
add_row("Soil", "All soil", "Total",
        annual_Gt(raw, raw["LULUCF_component"].isin(["organic_soil", "mineral_soil"])))
add_row("LULUCF", "All components", "Total",
        annual_Gt(raw, raw["analysis_layer"] == "LULUCF_net_flux__MgCO2e"))

df_wf = pd.DataFrame(rows_summary)

veg_total    = df_wf.loc[df_wf["Subsection"] == "All vegetation", "All_gases_Gt_CO2e_yr"].values[0]
soil_total   = df_wf.loc[df_wf["Subsection"] == "All soil",       "All_gases_Gt_CO2e_yr"].values[0]
lulucf_total = df_wf.loc[df_wf["Section"]    == "LULUCF",         "All_gases_Gt_CO2e_yr"].values[0]

def section_subtitle(val):
    term = "net sink" if val < 0 else "net source"
    return f"({term} = {val:.3g} Gt CO₂e yr⁻¹)"

section_subtitle_map = {
    "Vegetation": section_subtitle(veg_total),
    "Soil":       section_subtitle(soil_total),
    "LULUCF":     section_subtitle(lulucf_total),
}

# ── 1. Select detail rows + LULUCF total as closing bar ─────────────────────
detail_mask = (
    ((df_wf["Subsection"] == "Tall vegetation")   & (df_wf["Category"] != "Total")) |
    ( df_wf["Subsection"] == "Non-tall vegetation")                                  |
    ((df_wf["Subsection"] == "Organic soil")       & (df_wf["Category"] != "Total")) |
    ( df_wf["Subsection"] == "Mineral soil")                                         |
    ((df_wf["Section"]    == "LULUCF")             & (df_wf["Category"] == "Total"))
)
wf = df_wf[detail_mask][["Section", "Subsection", "Category", "All_gases_Gt_CO2e_yr"]].reset_index(drop=True)
wf["is_total"] = (wf["Section"] == "LULUCF") & (wf["Category"] == "Total")

# ── 2. Waterfall geometry ────────────────────────────────────────────────────
vals    = wf["All_gases_Gt_CO2e_yr"].values.astype(float)
bottoms = np.zeros(len(vals))
running = 0.0

for i in range(len(vals)):
    if wf["is_total"].iloc[i]:
        bottoms[i] = 0.0
    else:
        bottoms[i] = running
        running   += vals[i]

# ── 3. Colors ────────────────────────────────────────────────────────────────
SINK   = "#33cc33"
SOURCE = "#d279d2"
TOTAL  = "#616161"

colors = [
    TOTAL  if wf["is_total"].iloc[i] else
    SINK   if vals[i] < 0            else
    SOURCE
    for i in range(len(vals))
]

# ── 4. Tick labels ───────────────────────────────────────────────────────────
def make_label(row):
    if row["is_total"]:
        return "LULUCF\nTotal"
    cat, sub = row["Category"], row["Subsection"]
    short = {
        "Undisturbed tree cover (including fires)": "Undisturbed\ntree cover",
        "Partially disturbed tree cover":           "Partially\ndisturbed\ntree cover",
        "New tree cover":                           "New\ntree cover",
        "Tree cover loss":                          "Tree cover\nloss",
        "Fire":                                     "Organic soil\nFire",
        "Extraction":                               "Organic soil\nExtraction",
        "Drainage":                                 "Organic soil\nDrainage",
    }
    if cat == "All classes":
        return "Short veg. &\ncropland" if sub == "Non-tall vegetation" else sub
    return short.get(cat, cat)

labels = [make_label(row) for _, row in wf.iterrows()]

# ── 5. Section midpoints for top axis ───────────────────────────────────────
sections = wf["Section"].values
section_groups = {}
for i, s in enumerate(sections):
    section_groups.setdefault(s, []).append(i)

# ── 6. Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

bar_bottoms = np.where(vals < 0, bottoms + vals, bottoms)
bar_heights = np.abs(vals)

bars = ax.bar(
    range(len(vals)),
    bar_heights,
    bottom=bar_bottoms,
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    width=0.65,
)

ax.grid(False)
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# Dashed horizontal connectors between consecutive flow bars
for i in range(len(vals) - 1):
    if not wf["is_total"].iloc[i] and not wf["is_total"].iloc[i + 1]:
        y = bottoms[i] + vals[i]
        ax.plot([i + 0.33, i + 0.67], [y, y],
                color="grey", linewidth=0.8, linestyle="--", alpha=0.6)

# Bar value labels
SMALL_BAR = 0.6
for i, (bar, val) in enumerate(zip(bars, vals)):
    label_text = f"{val:+.2g}"
    if bar.get_height() < SMALL_BAR:
        if val >= 0:
            y_pos, va = bar.get_y() + bar.get_height() + 0.15, "bottom"
        else:
            y_pos, va = bar.get_y() - 0.15, "top"
        ax.text(bar.get_x() + bar.get_width() / 2, y_pos, label_text,
                ha="center", va=va, fontsize=12, color="black", fontweight="bold")
    else:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2, label_text,
                ha="center", va="center", fontsize=12, color="white", fontweight="bold")

# Zero reference line
ax.axhline(0, color="black", linewidth=0.9)

# Solid vertical section dividers
for i in range(1, len(sections)):
    if sections[i] != sections[i - 1]:
        ax.axvline(i - 0.5, color="#555555", linewidth=1.2, linestyle="-", alpha=0.35)

# Bottom axis
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labels, rotation=0, ha="center")
ax.tick_params(axis="x", labelsize=10)
ax.set_title("LULUCF GHG flux components (all GHGs, average for 2016-2024)", pad=20)

# Y-axis: no tick labels, 2.5 Gt gridlines, scale bar annotation
ax.yaxis.set_major_locator(MultipleLocator(2.5))
ax.set_yticklabels([])
ax.set_ylabel("")

trans = blended_transform_factory(ax.transAxes, ax.transData)
ax.annotate("", xy=(-0.03, 0), xytext=(-0.03, -2.5),
            xycoords=trans, textcoords=trans,
            annotation_clip=False,
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.0))
txt = ax.text(-0.05, -1.25, "2.5 Gt\nCO₂e yr⁻¹",
              transform=trans, ha="right", va="center", fontsize=9)
txt.set_clip_on(False)

# Top axis: section names (bold, full size) + smaller subtitles as separate text
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks([np.mean(idxs) for idxs in section_groups.values()])
ax2.set_xticklabels(list(section_groups.keys()), fontweight="bold")
ax2.tick_params(length=0, pad=18)   # pad lifts names up, leaving room for subtitle
ax2.grid(False)

# Subtitles slotted between top spine and section names
trans2 = blended_transform_factory(ax2.transData, ax2.transAxes)
for section, idxs in section_groups.items():
    t = ax2.text(
        np.mean(idxs), 1.01,
        section_subtitle_map[section],
        transform=trans2,
        ha="center", va="bottom",
        fontsize=9,
        clip_on=False,
    )

# Hide both top spines to remove double line
ax.spines["top"].set_visible(False)
ax2.spines["bottom"].set_visible(False)

# Legend
ax.legend(
    handles=[
        mpatches.Patch(color=SINK,   label="Sink (−)"),
        mpatches.Patch(color=SOURCE, label="Source (+)"),
        mpatches.Patch(color=TOTAL,  label="Net sink"),
    ],
    loc="lower right", fontsize=9
)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_global_waterfall_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()
df_wf

In [ ]:
### Global vegetation gross emissions timeseries by fire vs. non-fire, with and without Australia (single panel)
### Per Claude ("Create global emissions timeseries graph")

# ---- Setup ----
context_to_analyze = "land_state_node"

df_fig = LULUCF_outputs_dropped.copy()

# Filter to gross emissions (all gases)
df_emis = df_fig[df_fig["analysis_layer"] == f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"].copy()

# ---- Reclassify fire vs non-fire BEFORE aggregation ----
df_emis["fire_class"] = df_emis[context_to_analyze].apply(
    lambda x: "Fire" if re.search(r"9(0*)$", str(int(float(x)))) else "Non-fire"
)

# ---- Aggregate WITH Australia ----
df_with_Australia = (
    df_emis
    .groupby(["fire_class", "year"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_with_Australia["Australia_group"] = "with Australia"
df_with_Australia

# ---- Aggregate WITHOUT Australia ----
df_without_Australia = (
    df_emis[df_emis["country_name"] != "Australia"]
    .groupby(["fire_class", "year"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_without_Australia["Australia_group"] = "without Australia"

# Combine
df_emis_ts = pd.concat([df_with_Australia, df_without_Australia], ignore_index=True)

# Convert to Gt
df_emis_ts["value_Gt"] = df_emis_ts["flux_Mg_CO2e_yr"] / 1e9

# Create combined plotting label
df_emis_ts["series"] = df_emis_ts["fire_class"] + " (" + df_emis_ts["Australia_group"] + ")"
df_emis_ts

# ---- Plot ----
series_order = [
    "Non-fire (with Australia)",
    "Non-fire (without Australia)",
    "Fire (with Australia)",
    "Fire (without Australia)",
]

series_palette = {
    "Non-fire (with Australia)": "#1f77b4",
    "Non-fire (without Australia)": "#1f77b4",
    "Fire (with Australia)":     "#d62728",
    "Fire (without Australia)":  "#d62728",
}

series_dashes = {
    "Non-fire (with Australia)": "",
    "Non-fire (without Australia)": (5, 2),
    "Fire (with Australia)":     "",
    "Fire (without Australia)":  (5, 2),
}

plt.figure(figsize=(10, 5))

ax = sns.lineplot(
    data=df_emis_ts,
    x="year",
    y="value_Gt",
    hue="series",
    style="series",
    hue_order=series_order,
    style_order=series_order,
    palette=series_palette,
    dashes=series_dashes,
    linewidth=2.5
)

ax.axhline(0, color="black", linewidth=1)
ax.grid(False)

ax.set_xlabel(None)
ax.set_ylabel("Gt CO$_2$e yr$^{-1}$")
# ax.set_title("Gross emissions (all gases): fire vs non-fire, with and without Australia", pad=20)

ax.legend(
    title="Emission type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/veg_emis_timeseries_fire_Australia_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### Bar graphs of countries with largest net sources and sinks by continent
### Country-level gross/net LULUCF fluxes by region (horizontal stacked bars, net as dot)
### Top 3 net emitters + top 3 net removers per region + Other countries
### From Claude ('Bar graph of countries with largest net sources-sinks')

df_fig = LULUCF_outputs_dropped.copy()

n_years = len(cn.interval_end_years_annual)

# ── Layer definitions ───────────────────────────────────────────────────────

VEG_EMIS  = f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"
VEG_REMOV = f"veg_{cn.gross_removals_all_C_pools_pattern}"
NET_LAYER = "LULUCF_net_flux__MgCO2e"

ORGANIC_SOIL_LAYERS = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]
MINERAL_SOIL_LAYER = "SOC_change__mineral_soil_extent__0-30cm_MgCO2"
ORGANIC_LABEL = "organic_soil__all_gases__MgCO2e"

df_fig["analysis_layer"] = df_fig["analysis_layer"].replace(
    {l: ORGANIC_LABEL for l in ORGANIC_SOIL_LAYERS}
)

VEG_NET_LAYER = "veg_net_flux__all_C_pools__all_gases__MgCO2e"
LAYERS_NEEDED = [VEG_EMIS, VEG_REMOV, MINERAL_SOIL_LAYER, ORGANIC_LABEL, NET_LAYER, VEG_NET_LAYER]
df_fig = df_fig[df_fig["analysis_layer"].isin(LAYERS_NEEDED)].copy()

# ── Annual average flux per country × analysis layer ────────────────────────

df_country = (
    df_fig
    .groupby(["region_L1", "country_name", "analysis_layer"], as_index=False)
    .agg(flux_Mg=("flux_Mg_CO2e_yr", "sum"))
)
df_country["flux_Mt"] = df_country["flux_Mg"] / n_years / 1e6

df_wide = df_country.pivot_table(
    index=["region_L1", "country_name"],
    columns="analysis_layer",
    values="flux_Mt",
    aggfunc="sum",
    fill_value=0,
).reset_index()
df_wide.columns.name = None

for col in [VEG_EMIS, VEG_REMOV, MINERAL_SOIL_LAYER, ORGANIC_LABEL, NET_LAYER, VEG_NET_LAYER]:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

df_wide["gross_emis_Mt"]  = df_wide[VEG_EMIS] + df_wide[ORGANIC_LABEL] + df_wide[MINERAL_SOIL_LAYER]
df_wide["gross_remov_Mt"] = df_wide[VEG_REMOV]
df_wide["net_Mt"]         = df_wide[NET_LAYER]
df_wide["veg_net_Mt"]     = df_wide[VEG_NET_LAYER]

# ── Drop Antarctica and unassigned regions ────────────────────────────────────

REGIONS_TO_DROP = {"Antarctica", "Unassigned"}
df_wide = df_wide[~df_wide["region_L1"].isin(REGIONS_TO_DROP)].copy()

# ── Country selection ─────────────────────────────────────────────────────────

def select_countries(group, region_name):
    top_emis  = group.nlargest(3,  "net_Mt")["country_name"].tolist()
    top_remov = group.nsmallest(3, "net_Mt")["country_name"].tolist()
    featured  = list(dict.fromkeys(top_emis + top_remov))

    rows = []
    for cname in featured:
        r = group[group["country_name"] == cname].iloc[0].copy()
        r["region_L1"] = region_name
        r["featured_group"] = "source" if cname in top_emis else "sink"
        rows.append(r)

    other_mask = ~group["country_name"].isin(featured)
    if other_mask.any():
        other = group[other_mask].sum(numeric_only=True)
        other["country_name"] = "Other countries"
        other["region_L1"]       = region_name
        other["featured_group"] = "other"
        rows.append(other)

    return pd.DataFrame(rows)

df_plot = (
    df_wide
    .groupby("region_L1", group_keys=False)
    .apply(lambda grp: select_countries(grp, grp.name), include_groups=False)
    .reset_index(drop=True)
)

# ── Colour / style constants ──────────────────────────────────────────────────

COLOR_VEG_EMIS  = "#4d4d8f"
COLOR_ORG_EMIS  = "#7b7bbf"
COLOR_MIN_EMIS  = "#b3b3d9"
COLOR_VEG_REMOV = "#2d7d2d"
DOT_COLOR       = "black"
DOT_SIZE        = 21
VEG_DOT_SIZE    = DOT_SIZE * 0.7
X_TICKS         = [-3000, -2000, -1000, 0, 1000, 2000, 3000]

LEGEND_ORDER = [
    "Veg net flux",
    "LULUCF net flux",
    "Veg gross removals",
    "Veg gross emissions",
    "Organic soil emissions",
    "Mineral soil emissions",
]

# ── Build the figure ──────────────────────────────────────────────────────────

regions = sorted(df_plot["region_L1"].unique())
n_regions = len(regions)
ncols = 2
nrows = (n_regions + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5), squeeze=False)

x_abs_max = max(abs(v) for v in X_TICKS)
x_pad = x_abs_max * 0.06
xlim = (-x_abs_max - x_pad, x_abs_max + x_pad)

_row_mids = {}

for idx, region in enumerate(regions):
    row = idx // ncols
    col = idx % ncols
    ax  = axes[row][col]
    is_bottom      = (row == nrows - 1)
    is_right_panel = (col == 1)

    df_r = df_plot[df_plot["region_L1"] == region].copy()
    other_row  = df_r[df_r["featured_group"] == "other"]
    emis_r     = df_r[df_r["featured_group"] == "source"].sort_values("net_Mt", ascending=False)
    remov_r    = df_r[df_r["featured_group"] == "sink"].sort_values("net_Mt", ascending=False)
    feature_r  = pd.concat([emis_r, remov_r], ignore_index=True)
    df_r       = pd.concat([other_row, feature_r], ignore_index=True)

    countries = df_r["country_name"].tolist()
    y = np.arange(len(countries))

    veg_e = df_r[VEG_EMIS].values
    org_e = df_r[ORGANIC_LABEL].values
    min_e = df_r[MINERAL_SOIL_LAYER].values

    ax.barh(y, veg_e,                   height=0.6, color=COLOR_VEG_EMIS,  label="Veg gross emissions")
    ax.barh(y, org_e, left=veg_e,       height=0.6, color=COLOR_ORG_EMIS,  label="Organic soil emissions")
    ax.barh(y, min_e, left=veg_e+org_e, height=0.6, color=COLOR_MIN_EMIS,  label="Mineral soil emissions")
    ax.barh(y, df_r["gross_remov_Mt"].values, height=0.6,
            color=COLOR_VEG_REMOV, label="Veg gross removals")
    ax.scatter(df_r["net_Mt"].values, y, color=DOT_COLOR, zorder=5, s=DOT_SIZE, label="LULUCF net flux")
    ax.scatter(df_r["veg_net_Mt"].values, y, color="#888888", zorder=5, s=VEG_DOT_SIZE, label="Veg net flux", marker="o")

    # Dashed lines: between source/sink groups, and between Other and sources
    n_emis = len(emis_r)
    if n_emis > 0 and len(remov_r) > 0:
        ax.axhline(y=n_emis + 0.5, color="gray", linestyle="--", linewidth=0.8, zorder=3)
    if n_emis > 0:
        ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, zorder=3)
    if col == 0:
        _row_mids[row] = (
            (1 + n_emis) / 2 if n_emis > 0 else None,
            n_emis + 0.5 + len(remov_r) / 2 if len(remov_r) > 0 else None,
            ax,
        )

    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.tick_params(axis="y", left=False, right=False)
    x_label = xlim[0] + (xlim[1] - xlim[0]) * 0.01
    for yi, name in zip(y, countries):
        ax.text(x_label, yi, name, ha="left", va="center", fontsize=8, clip_on=True)
    ax.set_xlim(xlim)
    ax.set_xticks(X_TICKS)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(region, fontsize=10, fontweight="bold")
    ax.grid(axis="x", linewidth=0.5, alpha=0.5, which="major")
    ax.set_axisbelow(True)

    if is_bottom:
        ax.set_xlabel("Average annual flux (MtCO₂e yr⁻¹)", fontsize=7)
        ax.tick_params(axis="x", labelsize=7)
    else:
        ax.set_xlabel(None)
        ax.tick_params(axis="x", labelbottom=False)

for idx in range(n_regions, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

# ── Legend in specified order ─────────────────────────────────────────────────

handles, labels = axes[0][0].get_legend_handles_labels()
label_to_handle = dict(zip(labels, handles))
ordered_handles = [label_to_handle[l] for l in LEGEND_ORDER if l in label_to_handle]
ordered_labels  = [l for l in LEGEND_ORDER if l in label_to_handle]

fig.legend(
    ordered_handles, ordered_labels,
    loc="lower center",
    ncol=5,
    frameon=False,
    fontsize=12,
    bbox_to_anchor=(0.5, -0.04),
)

# fig.suptitle("Average annual LULUCF fluxes by region and country", fontsize=13, y=1.01)
plt.tight_layout()

for row_idx, (y_src, y_snk, ax_left) in _row_mids.items():
    if not axes[row_idx][1].get_visible():
        continue
    pos_l = ax_left.get_position()
    pos_r = axes[row_idx][1].get_position()
    x_mid = (pos_l.x1 + pos_r.x0) / 2
    for y_dat, lbl in [(y_src, "Largest sources"), (y_snk, "Largest sinks")]:
        if y_dat is None:
            continue
        _, y_fig = fig.transFigure.inverted().transform(
            ax_left.transData.transform([0, y_dat])
        )
        fig.text(x_mid, y_fig, lbl, ha="center", va="center",
                 fontsize=7, color="dimgray", style="italic", rotation=90)

# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_region_country_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# # To get stats for checking
# print(LULUCF_outputs_dropped['analysis_layer'].unique())
#
# print(LULUCF_outputs_dropped[
#     (LULUCF_outputs_dropped["adm0"] == "AUS") &
#     (LULUCF_outputs_dropped["analysis_layer"] == "organic_soil_drainage__all_gases__MgCO2e")
# ]["flux_Mg_CO2e_yr"].sum()/9/10**9)
#
# df_wide[df_wide["region_L1"] == "South America"][["country_name", "net_Mt"]].sort_values("net_Mt")

In [ ]:
### Pan et al. 2024-style regional map: average annual LULUCF component fluxes by UN geoscheme region
### From Claude ('Pan et al. regional LULUCF flux map')

VEG_GROSS_EMIS    = f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"
VEG_GROSS_REMOV   = f"veg_{cn.gross_removals_all_C_pools_pattern}"
VEG_NET_LAYER     = "veg_net_flux__all_C_pools__all_gases__MgCO2e"
MINERAL_SOIL_LAYER = "SOC_change__mineral_soil_extent__0-30cm_MgCO2"
ORGANIC_SOIL_LAYERS = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]
NET_LULUCF_LAYER = "LULUCF_net_flux__MgCO2e"

ALL_LAYERS = (
    [VEG_GROSS_EMIS, VEG_GROSS_REMOV, VEG_NET_LAYER, MINERAL_SOIL_LAYER]
    + ORGANIC_SOIL_LAYERS + [NET_LULUCF_LAYER]
)
LAYER_TO_COMP = {
    VEG_GROSS_EMIS:     "veg_gross_emis",
    VEG_GROSS_REMOV:    "veg_gross_remov",
    VEG_NET_LAYER:      "veg_net",
    MINERAL_SOIL_LAYER: "mineral_soil",
    **{l: "organic_soil" for l in ORGANIC_SOIL_LAYERS},
    NET_LULUCF_LAYER:   "total_LULUCF",
}

df_map = LULUCF_outputs_dropped[LULUCF_outputs_dropped["analysis_layer"].isin(ALL_LAYERS)].copy()
df_map["plot_component"] = df_map["analysis_layer"].map(LAYER_TO_COMP)
df_map["map_region"] = np.where(
    df_map["region_L1"] == "Oceania", "Oceania", df_map["region_L2_L3"]
)
df_map = df_map[
    df_map["map_region"].notna() & ~df_map["map_region"].isin(["Antarctica", "Unassigned"])
]
regional_avg = (
    df_map
    .groupby(["map_region", "plot_component", "year"])["flux_Mg_CO2e_yr"]
    .sum()
    .groupby(level=["map_region", "plot_component"])
    .mean()
    .reset_index(name="avg_flux")
)
regional_wide = regional_avg.pivot(
    index="map_region", columns="plot_component", values="avg_flux"
)

ROBINSON_CRS = cn.Robinson_crs

def _iso_to_map_region(iso):
    if not isinstance(iso, str) or iso == "-99":
        return None
    if cn.iso_to_region_UN_geoscheme_L1.get(iso) == "Oceania":
        return "Oceania"
    return cn.iso_to_region_UN_geoscheme_L2_L3.get(iso)

try:
    world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
except AttributeError:
    _ne_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(_ne_url)
    world = world.rename(columns={"ISO_A3_EH": "iso_a3"})

world = world[world["iso_a3"] != "ATA"].copy()
world["map_region"] = world["iso_a3"].map(_iso_to_map_region)

all_regions_sorted = sorted([
    "Northern Africa", "Western Africa", "Middle Africa", "Eastern Africa", "Southern Africa",
    "Northern Europe", "Western Europe", "Southern Europe", "Eastern Europe",
    "Western Asia", "Central Asia", "Southern Asia", "Eastern Asia", "South-eastern Asia",
    "Northern America", "Central America", "Caribbean", "South America", "Oceania",
])
_cmap = plt.colormaps["tab20"].resampled(len(all_regions_sorted))
REGION_COLORS = {region: mcolors.to_hex(_cmap(i)) for i, region in enumerate(all_regions_sorted)}

world["map_color"] = world["map_region"].map(lambda r: REGION_COLORS.get(r, "#d9d9d9"))
world = world.to_crs(ROBINSON_CRS)

BAR_POSITIONS_WGS84 = {
    "Northern Africa":    ( 15,  27),
    "Western Africa":     (-12,  10),
    "Middle Africa":      ( 22,  -3),
    "Eastern Africa":     ( 40,   2),
    "Southern Africa":    ( 27, -27),
    "Northern Europe":    ( 18,  68),
    "Western Europe":     (-12,  48),
    "Southern Europe":    ( 18,  40),
    "Eastern Europe":     ( 38,  55),
    "Western Asia":       ( 45,  24),
    "Central Asia":       ( 68,  44),
    "Southern Asia":      ( 82,  22),
    "Eastern Asia":       (105,  35),
    "South-eastern Asia": (120,   5),
    "Northern America":   (-100,  56),
    "Central America":    ( -92,  14),
    "Caribbean":          ( -60,  20),
    "South America":      ( -62, -12),
    "Oceania":            ( 152, -25),
}

_anchor_gdf = gpd.GeoDataFrame(
    {"region": list(BAR_POSITIONS_WGS84.keys())},
    geometry=[Point(lon, lat) for lon, lat in BAR_POSITIONS_WGS84.values()],
    crs="EPSG:4326",
).to_crs(ROBINSON_CRS)

BAR_POSITIONS = {
    row["region"]: (row.geometry.x, row.geometry.y)
    for _, row in _anchor_gdf.iterrows()
}

# Change 3: match gross veg colors to timeseries color_map (cell 35)
COMP_COLORS = {
    "veg_gross_emis":  "#d279d2",   # light purple-pink, same as timeseries
    "veg_gross_remov": "#33cc33",   # bright green, same as timeseries
    "organic_soil":    "#993399",
    "mineral_soil":    "#8c564b",
}
VEG_NET_EDGE_COLOR = "#1a6b3a"

# Change 4: narrow bars by 20% (250_000 → 200_000)
SUB_HW   = 200_000
HALF_GAP = 50_000

comp_cols_for_scale = ["veg_gross_emis", "veg_gross_remov", "organic_soil", "mineral_soil"]
max_abs = regional_wide[comp_cols_for_scale].abs().max().max()
SCALE   = 3_000_000 / max_abs

fig, ax = plt.subplots(figsize=(22, 11))
ax.set_aspect("equal")
ax.set_axis_off()

bounds = world.total_bounds
ax.set_xlim(bounds[0], bounds[2])
ax.set_ylim(bounds[1], bounds[3])

world.plot(ax=ax, color=world["map_color"].tolist(), edgecolor="white", linewidth=0.3, zorder=1)

for region, row in regional_wide.iterrows():
    if region not in BAR_POSITIONS:
        continue
    cx, cy = BAR_POSITIONS[region]
    cx_veg  = cx - SUB_HW - HALF_GAP
    cx_soil = cx + SUB_HW + HALF_GAP
    veg_left   = cx_veg  - SUB_HW
    veg_right  = cx_veg  + SUB_HW
    soil_left  = cx_soil - SUB_HW
    soil_right = cx_soil + SUB_HW

    veg_pos_top = veg_neg_bot = 0.0
    for comp in ["veg_gross_emis", "veg_gross_remov"]:
        val = row.get(comp, np.nan)
        if pd.isna(val): continue
        h = val * SCALE
        bottom = veg_pos_top if val >= 0 else veg_neg_bot
        if val >= 0: veg_pos_top += h
        else: veg_neg_bot += h
        ax.add_patch(mpatches.Rectangle(
            (veg_left, cy + bottom), SUB_HW * 2, h,
            facecolor=COMP_COLORS[comp], edgecolor="none", zorder=4,
        ))

    soil_pos_top = soil_neg_bot = 0.0
    for comp in ["organic_soil", "mineral_soil"]:
        val = row.get(comp, np.nan)
        if pd.isna(val): continue
        h = val * SCALE
        bottom = soil_pos_top if val >= 0 else soil_neg_bot
        if val >= 0: soil_pos_top += h
        else: soil_neg_bot += h
        ax.add_patch(mpatches.Rectangle(
            (soil_left, cy + bottom), SUB_HW * 2, h,
            facecolor=COMP_COLORS[comp], edgecolor="none", zorder=4,
        ))

    ax.plot([veg_left, soil_right], [cy, cy], color="black", linewidth=2.0, zorder=5)

    veg_net = row.get("veg_net", np.nan)
    if not pd.isna(veg_net):
        ax.scatter(cx_veg, cy + veg_net * SCALE, s=30, zorder=6,
                   facecolors="white", edgecolors=VEG_NET_EDGE_COLOR, linewidths=1.5)

    lulucf_net = row.get("total_LULUCF", np.nan)
    if not pd.isna(lulucf_net):
        ax.scatter(cx, cy + lulucf_net * SCALE, s=35, color="black", zorder=6)

    overall_neg_bot = min(veg_neg_bot, soil_neg_bot, 0)
    ax.text(cx, cy + overall_neg_bot - 150_000, region,
            ha="center", va="top", fontsize=7.8, zorder=7)

legend_handles = [
    mpatches.Patch(facecolor=COMP_COLORS["veg_gross_emis"],  label="Vegetation gross emissions"),
    mpatches.Patch(facecolor=COMP_COLORS["veg_gross_remov"], label="Vegetation gross removals"),
    plt.Line2D([0], [0], marker="o", color="w", markersize=7,
               markerfacecolor="white", markeredgecolor=VEG_NET_EDGE_COLOR,
               markeredgewidth=1.5, label="Vegetation net flux"),
    mpatches.Patch(facecolor=COMP_COLORS["organic_soil"],    label="Organic soil emissions"),
    mpatches.Patch(facecolor=COMP_COLORS["mineral_soil"],    label="Mineral soil net flux"),
    plt.Line2D([0], [0], marker="o", color="w", markersize=7,
               markerfacecolor="black", label="LULUCF net flux"),
]
ax.legend(handles=legend_handles,
          bbox_to_anchor=(bounds[0] + 4_200_000, bounds[1] + 300_000),
          bbox_transform=ax.transData, loc="lower left",
          fontsize=9, framealpha=0.9, borderpad=0.7)

# Change 1: scale box moved right of legend (legend at bounds[0]+4_200_000)
sb_x  = bounds[0] + 8_100_000
sb_y  = bounds[1] + 320_000
ref_w = SUB_HW * 1.5
ref_h_500  = 500e6  * SCALE
ref_h_1000 = 1000e6 * SCALE

# Change 2: title above scale box
ax.text(sb_x + ref_w / 2, sb_y + ref_h_1000 + 150_000,
        "Average annual flux, \n2016–2024",
        ha="left", va="bottom", fontsize=7.5, zorder=8)

ax.add_patch(mpatches.Rectangle(
    (sb_x, sb_y), ref_w, ref_h_1000,
    facecolor="#cccccc", edgecolor="#888888", linewidth=0.5, zorder=7,
))
ax.plot([sb_x, sb_x + ref_w], [sb_y + ref_h_500, sb_y + ref_h_500],
        color="#888888", linewidth=0.8, zorder=8)

tick_x = sb_x + ref_w + 100_000
ax.text(tick_x, sb_y,              "0",                  va="center", fontsize=7.5, zorder=8)
ax.text(tick_x, sb_y + ref_h_500,  "500 Mt",             va="center", fontsize=7.5, zorder=8)
ax.text(tick_x, sb_y + ref_h_1000, "1000 Mt CO₂e yr⁻¹", va="center", fontsize=7.5, zorder=8)

# ax.set_title("Average annual LULUCF flux components by region", fontsize=13, pad=10)
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/regional_map_flux_bar_graphs_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### Average annual fluxes bar chart: gross emissions | gross removals | net flux
### With Claude session "Annual net fluxes bar graph"

# ── Colors ────────────────────────────────────────────────────────────────────
C_EMIS_NONTALL = "#e8b0e8"   # light pink   — short veg & cropland
C_EMIS_DISTURB = "#d279d2"   # medium pink  — partially disturbed
C_EMIS_LOSS    = "#8b008b"   # dark magenta — tree cover loss
C_MIN_SOIL     = "#8c564b"   # brown        — mineral soil
C_ORG_SOIL     = "#993399"   # dark purple  — organic soil (combined)

C_REMOV_UNDIST = "#006600"   # dark green   — undisturbed tree cover
C_REMOV_NEW    = "#33cc33"   # medium green — new tree cover

C_NET          = "#616161"   # gray         — LULUCF net flux

# ── Data aggregation ──────────────────────────────────────────────────────────
_gross = LULUCF_outputs_dropped[
    LULUCF_outputs_dropped["analysis_layer"].str.contains("gross_emissions|gross_removals")
].copy()
_net = LULUCF_outputs_dropped[
    ~LULUCF_outputs_dropped["analysis_layer"].str.contains("gross_emissions|gross_removals")
].copy()

_n_yrs_net = _net["year"].nunique()

_emis_layer  = f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"
_remov_layer = f"veg_{cn.gross_removals_all_C_pools_pattern}"

_n_yrs_emis  = _gross.loc[_gross["analysis_layer"] == _emis_layer,  "year"].nunique()
_n_yrs_remov = _gross.loc[_gross["analysis_layer"] == _remov_layer, "year"].nunique()

def _Gt(df, mask, n_yrs):
    return df.loc[mask, "flux_Mg_CO2e_yr"].sum() / n_yrs / 1e9

_e   = _gross["analysis_layer"] == _emis_layer
_r   = _gross["analysis_layer"] == _remov_layer
dlc  = _gross["land_state_detailed_class"]

# Gross emissions
veg_emis_nontall_Gt = _Gt(_gross, _e & (_gross["land_state_broad_class"] != "tree"),                        _n_yrs_emis)
veg_emis_disturb_Gt = _Gt(_gross, _e & dlc.isin(["tree_tree_disturbed", "tree_tree_disturbed_fire_only"]),  _n_yrs_emis)
veg_emis_loss_Gt    = _Gt(_gross, _e & (dlc == "tree_loss"),                                                _n_yrs_emis)
mineral_soil_Gt     = _Gt(_net,   _net["LULUCF_component"] == "mineral_soil",                              _n_yrs_net)
org_soil_Gt         = _Gt(_net,   _net["LULUCF_component"] == "organic_soil",                              _n_yrs_net)

# Gross removals
veg_remov_undist_Gt = _Gt(_gross, _r & (dlc == "tree_tree_undisturbed"), _n_yrs_remov)
veg_remov_new_Gt    = _Gt(_gross, _r & (dlc == "tree_gain"),             _n_yrs_remov)

# Net flux
lulucf_net_Gt = _Gt(_net, _net["analysis_layer"] == "LULUCF_net_flux__MgCO2e", _n_yrs_net)

# QC print
for lbl, val in [
    ("Emis: short veg & cropland",  veg_emis_nontall_Gt),
    ("Emis: partially disturbed",   veg_emis_disturb_Gt),
    ("Emis: tree cover loss",       veg_emis_loss_Gt),
    ("Mineral soil",                mineral_soil_Gt),
    ("Organic soil (combined)",     org_soil_Gt),
    ("Remov: undisturbed",          veg_remov_undist_Gt),
    ("Remov: new tree cover",       veg_remov_new_Gt),
    ("LULUCF net flux",             lulucf_net_Gt),
]:
    print(f"  {lbl:<30s}: {val:+.2f} Gt CO2e/yr")

# ── Bar positions ─────────────────────────────────────────────────────────────
bar_w = 0.6
bsp   = 0.9   # bar spacing within section (center-to-center)

# Left section: gross emissions (5 bars)
x_en  = 0.0              # short veg & cropland
x_ed  = x_en + bsp       # partially disturbed
x_el  = x_ed + bsp       # tree cover loss
x_ms  = x_el + bsp       # mineral soil
x_os  = x_ms + bsp       # organic soil

# Middle section: gross removals (2 bars)
x_ru  = x_os + 1.8       # undisturbed
x_rn  = x_ru + bsp       # new tree cover

# Right section: net flux (1 bar)
x_nf  = x_rn + 1.8

# Section dividers
div1  = (x_os + x_ru) / 2
div2  = (x_rn + x_nf) / 2

# ── Chart ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

x_emis, x_remov, x_net = 0.0, 2.5, 4.5
bar_w = 0.7

# Gross emissions — stacked upward
ax.bar(x_emis, veg_emis_nontall_Gt, width=bar_w, color=C_EMIS_NONTALL, zorder=2)
ax.bar(x_emis, veg_emis_disturb_Gt, width=bar_w, color=C_EMIS_DISTURB,
       bottom=veg_emis_nontall_Gt, zorder=2)
ax.bar(x_emis, veg_emis_loss_Gt,    width=bar_w, color=C_EMIS_LOSS,
       bottom=veg_emis_nontall_Gt + veg_emis_disturb_Gt, zorder=2)
ax.bar(x_emis, mineral_soil_Gt,     width=bar_w, color=C_MIN_SOIL,
       bottom=veg_emis_nontall_Gt + veg_emis_disturb_Gt + veg_emis_loss_Gt, zorder=2)
ax.bar(x_emis, org_soil_Gt,         width=bar_w, color=C_ORG_SOIL,
       bottom=veg_emis_nontall_Gt + veg_emis_disturb_Gt + veg_emis_loss_Gt + mineral_soil_Gt, zorder=2)

# Gross removals — stacked downward
ax.bar(x_remov, veg_remov_undist_Gt, width=bar_w, color=C_REMOV_UNDIST, zorder=2)
ax.bar(x_remov, veg_remov_new_Gt,    width=bar_w, color=C_REMOV_NEW,
       bottom=veg_remov_undist_Gt, zorder=2)

# Net flux
ax.bar(x_net, lulucf_net_Gt, width=bar_w, color=C_NET, zorder=2)

# Zero line and section dividers
ax.axhline(0, color="black", linewidth=0.8, zorder=1)
ax.axvline((x_emis + x_remov) / 2, color="black", linewidth=1.2, zorder=1)
ax.axvline((x_remov + x_net)  / 2, color="black", linewidth=1.2, zorder=1)

# Axes formatting
ax.set_xticks([x_emis, x_remov, x_net])
ax.set_xticklabels(["Gross\nemissions", "Gross\nremovals", "LULUCF\nnet flux"])
ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
ax.set_title(
    f"Average annual LULUCF GHG fluxes, "
    f"{cn.interval_end_years_annual[0]}–{cn.interval_end_years_annual[-1]}",
    pad=10,
)
ax.set_xlim(x_emis - bar_w, x_net + bar_w)
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", length=0, labelsize=12)

# Legend — right of figure
legend_handles = [
    mpatches.Patch(color=C_ORG_SOIL,     label="Organic soil"),
    mpatches.Patch(color=C_MIN_SOIL,     label="Mineral soil"),
    mpatches.Patch(color=C_EMIS_LOSS,    label="Tree cover loss"),
    mpatches.Patch(color=C_EMIS_DISTURB, label="Partially disturbed"),
    mpatches.Patch(color=C_EMIS_NONTALL, label="Short veg & cropland"),
    mpatches.Patch(color=C_REMOV_UNDIST, label="Undisturbed tree cover"),
    mpatches.Patch(color=C_REMOV_NEW,    label="New tree cover"),
    mpatches.Patch(color=C_NET,          label="LULUCF net flux"),
]
ax.legend(handles=legend_handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
          frameon=False, fontsize=9)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/average_annual_bar_graphs_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### Global Carbon Budget comparison bar graphs.
### Uses an Excel spreadsheet in which I've already aggregated the GCB values and calculated the LCL values (i.e. translated the LCL values to GCB).
### So, the actual workflow for this is: create LULUCF pandas dataframe from flox zonal stats runs -> ingest into Postgres table -> query postgres table and put results in Excel -> perform GCB translation -> run this script
### I decided it was easier to calculate translated LULUCF framework values and write notes about GCB and LCL data in Excel than doing all of it in a Jupyter notebook. Hence, this route that goes through Excel. 
### With Claude session 'LULUCF vs Global Carbon Budget graphs'

EXCEL_PATH = (
    '/mnt/c/Users/David.Gibbs/OneDrive - World Resources Institute/Documents/Projects'
    '/AFOLU_flux_model__all_land_all_carbon/LULUCF/Global_Carbon_Budget_comparison'
    '/LCL_LULUCF_vs_GCB_land_20260523.xlsx'
)
GCB_COLOR = '#808080'
LCL_COLOR = '#D4A017'

raw = pd.read_excel(EXCEL_PATH, sheet_name='comparison', header=0)
raw = raw.iloc[:, [1, 4, 5, 12, 13, 16]].copy()
raw.columns = ['gcb_label', 'gcb_flux', 'gcb_stdev', 'lcl_label', 'lcl_flux', 'lcl_stdev']

def ix(excel_rows):
    return [r - 2 for r in excel_rows]

def wrap_label(s, width=16):
    return '\n'.join(textwrap.wrap(str(s), width=width))

def parse_stdev(val, central=None):
    if pd.isna(val):
        return np.nan, np.nan
    s = str(val).strip()
    if s.upper() == 'N/A':
        return np.nan, np.nan
    m = re.match(r'\[\s*([+-]?\d*\.?\d+)\s*,\s*([+-]?\d*\.?\d+)\s*\]', s)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        if central is not None:
            return abs(float(central) - lo), abs(hi - float(central))
        return np.nan, np.nan
    try:
        v = float(s)
        return abs(v), abs(v)
    except ValueError:
        return np.nan, np.nan

def add_errorbars(ax, x, y, lo, hi):
    if not (np.isnan(lo) or np.isnan(hi)):
        ax.errorbar(x, y, yerr=[[lo], [hi]], fmt='none',
                    ecolor='black', capsize=4, elinewidth=1.5)

legend_kw = dict(loc='lower left', fontsize=6, handlelength=1.0,
                 handleheight=0.6, borderpad=0.4, labelspacing=0.2)
legend_handles = [
    mpatches.Patch(color=GCB_COLOR, label='GCB'),
    mpatches.Patch(color=LCL_COLOR, label='LCL LULUCF'),
]

# ── Data prep ─────────────────────────────────────────────────────────────────
gcb_L = raw.iloc[ix([2, 3, 4, 5, 7, 8, 9])].reset_index(drop=True)
lcl_L = raw.iloc[ix([8, 9])].reset_index(drop=True)

bars_L = []
for i in range(6):
    bars_L.append({'label': gcb_L.iloc[i]['gcb_label'], 'flux': gcb_L.iloc[i]['gcb_flux'],
                   'stdev': gcb_L.iloc[i]['gcb_stdev'], 'color': GCB_COLOR})
bars_L.append({'label': lcl_L.iloc[0]['lcl_label'], 'flux': lcl_L.iloc[0]['lcl_flux'],
               'stdev': lcl_L.iloc[0]['lcl_stdev'], 'color': LCL_COLOR})
no_organic_start = len(bars_L)   # = 7
bars_L.append({'label': gcb_L.iloc[6]['gcb_label'], 'flux': gcb_L.iloc[6]['gcb_flux'],
               'stdev': gcb_L.iloc[6]['gcb_stdev'], 'color': GCB_COLOR})
bars_L.append({'label': lcl_L.iloc[1]['lcl_label'], 'flux': lcl_L.iloc[1]['lcl_flux'],
               'stdev': lcl_L.iloc[1]['lcl_stdev'], 'color': LCL_COLOR})

mid_included  = (no_organic_start - 1) / 2        # centre of positions 0–6  = 3.0
mid_excluded  = (no_organic_start + len(bars_L) - 1) / 2   # centre of 7–8 = 7.5

both_R = raw.iloc[ix([11, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25])].reset_index(drop=True)
x_c = np.arange(len(both_R))
bw  = 0.4

# ── Panel draw functions ──────────────────────────────────────────────────────
def draw_left(ax):
    ax.yaxis.grid(True, linewidth=0.5, color='lightgrey')
    ax.set_axisbelow(True)
    for i, bar in enumerate(bars_L):
        lo, hi = parse_stdev(bar['stdev'], bar['flux'])
        ax.bar(i, bar['flux'], color=bar['color'])
        add_errorbars(ax, i, bar['flux'], lo, hi)

    # Section divider and labels (xaxis transform: data-x, axes-fraction-y)
    ax.axvline(x=no_organic_start - 0.5, color='gray', linewidth=1.0)
    ax.text(mid_included, 0.97, 'Organic soil included',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())
    ax.text(mid_excluded, 0.97, 'Organic soil excluded',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())

    ax.set_xticks(np.arange(len(bars_L)))
    ax.set_xticklabels([wrap_label(bar['label']) for bar in bars_L],
                       rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gt CO₂ yr⁻¹')
    ax.set_title('Net atmosphere-land flux comparison\n(GCB: 2015–2024; LCL 2016–2024)')
    ax.legend(handles=legend_handles, **legend_kw)
    yabs = max(abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1]))
    ax.set_ylim(-yabs, yabs)

def draw_right(ax):
    ax.yaxis.grid(True, linewidth=0.5, color='lightgrey')
    ax.set_axisbelow(True)
    ax.bar(x_c - bw / 2, both_R['gcb_flux'], width=bw, color=GCB_COLOR)
    for i, row in both_R.iterrows():
        lo, hi = parse_stdev(row['gcb_stdev'], row['gcb_flux'])
        add_errorbars(ax, x_c[i] - bw / 2, row['gcb_flux'], lo, hi)
    ax.bar(x_c + bw / 2, both_R['lcl_flux'], width=bw, color=LCL_COLOR)
    for i, row in both_R.iterrows():
        lo, hi = parse_stdev(row['lcl_stdev'], row['lcl_flux'])
        add_errorbars(ax, x_c[i] + bw / 2, row['lcl_flux'], lo, hi)

    # Section divider and labels
    s_land_x = x_c[-1]                      # position of S_land bar group = 10
    mid_eluc  = (x_c[0] + x_c[-2]) / 2     # centre of E_luc bars (0–9) = 4.5
    ax.axvline(x=s_land_x - 0.5, color='gray', linewidth=1.0)
    ax.text(mid_eluc, 0.97, 'E_luc',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())
    ax.text(s_land_x, 0.97, 'S_land',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())

    ax.set_xticks(x_c)
    ax.set_xticklabels([wrap_label(l) for l in both_R['gcb_label']],
                       rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gt CO₂ yr⁻¹')
    ax.set_title('E_luc and S_land comparison\n(GCB: 2015–2024; LCL 2016–2024)')
    ax.legend(handles=legend_handles, **legend_kw)
    yabs = max(abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1]))
    ax.set_ylim(-yabs, yabs)

# ── Individual figure a ───────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(12, 6))
draw_left(ax1)
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_GCB_land_atmos_flux_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# ── Individual figure b ───────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 6))
draw_right(ax2)
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_GCB_Eluc_Sland_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# ── Combined figure (a above b) ───────────────────────────────────────────────
fig_c, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(14, 12))
draw_left(ax_a)
draw_right(ax_b)
for ax, label in zip([ax_a, ax_b], ['a', 'b']):
    ax.text(0.01, 0.98, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_GCB_two_panel_comparison_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# ── Print values ──────────────────────────────────────────────────────────────
def fmt_err(val, central=None):
    lo, hi = parse_stdev(val, central)
    if np.isnan(lo):
        return 'N/A'
    return f'±{lo:.2f}' if lo == hi else f'+{hi:.2f}/-{lo:.2f}'

print('=== Panel a: Net atmosphere-land flux ===')
print(f'\n{"Label":<45} {"Source":>6} {"Flux":>10} {"Uncertainty":>20}')
print('-' * 83)
for bar in bars_L:
    src = 'GCB' if bar['color'] == GCB_COLOR else 'LCL'
    print(f'{str(bar["label"]):<45} {src:>6} {bar["flux"]:>10.2f} {fmt_err(bar["stdev"], bar["flux"]):>20}')

print('\n=== Panel b: E_luc and S_land ===')
print(f'\n{"Label":<45} {"GCB flux":>10} {"GCB uncert":>15} {"LCL flux":>10} {"LCL uncert":>15}')
print('-' * 97)
for _, row in both_R.iterrows():
    print(f'{str(row["gcb_label"]):<45} {row["gcb_flux"]:>10.2f} '
          f'{fmt_err(row["gcb_stdev"], row["gcb_flux"]):>15} '
          f'{row["lcl_flux"]:>10.2f} {fmt_err(row["lcl_stdev"], row["lcl_flux"]):>15}')